In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

import os
from pathlib import Path
import xarray as xr
import copernicusmarine

# =========================
# PROJECT CONFIG
# =========================

ROOT = "/content/drive/MyDrive/OceanEmbed"

GLORYS_DIR = Path(ROOT) / "raw" / "GLORYS"
GLORYS_DIR.mkdir(parents=True, exist_ok=True)

GLORYS_DATASET = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

# Required region
MIN_LON = 45
MAX_LON = 105
MIN_LAT = 5
MAX_LAT = 30

# Download enough depth to capture native levels around 1000 m
DEPTH_MIN = 0
DEPTH_MAX = 1200

YEARS = [2005, 2006, 2007, 2008, 2009]

print("GLORYS directory:")
print(GLORYS_DIR)

print("\nExisting files:")
for f in sorted(GLORYS_DIR.glob("*.nc")):
    print(f"{f.name:40s} {f.stat().st_size / (1024**3):.2f} GB")

GLORYS directory:
/content/drive/MyDrive/OceanEmbed/raw/GLORYS

Existing files:


In [ ]:
!pip install -q copernicusmarine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# OceanEmbed - ALL DATASET DOWNLOADER
# Google Colab + Google Drive
#
# TRAIN : 2000-2004
# TEST  : 2005
#
# Region:
# Latitude  : 5N -> 30N
# Longitude : 45E -> 105E
#
# GLORYS depth:
# 0 -> 1200 m
# so that 1000 m target can be interpolated later
#
# Downloads:
#   GLORYS thetao
#   SST
#   SSS
#   SLA
#   Surface currents U/V
#   Wind ASC U/V
#   Wind DES U/V
#
# ARGO is NOT downloaded here.
# ============================================================


# ============================================================
# 1. INSTALL PACKAGE
# ============================================================

!pip install -q copernicusmarine


# ============================================================
# 2. IMPORTS
# ============================================================

import os
from getpass import getpass
import copernicusmarine


# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)


# ============================================================
# 4. CREATE PROJECT FOLDERS
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"

RAW = f"{ROOT}/raw"

DIRS = [
    "GLORYS",
    "SST",
    "SSS",
    "SLA",
    "CURRENT",
    "WIND",
    "ARGO"
]

for d in DIRS:
    os.makedirs(
        f"{RAW}/{d}",
        exist_ok=True
    )

print("=" * 70)
print("OceanEmbed folders created")
print("=" * 70)

print("ROOT:")
print(ROOT)

print("\nRAW folders:")

for d in DIRS:
    print(f"  {RAW}/{d}")


# ============================================================
# 5. COPERNICUS MARINE LOGIN
# ============================================================

print("\n" + "=" * 70)
print("COPERNICUS MARINE LOGIN")
print("=" * 70)

USERNAME = input("Copernicus username: ")
PASSWORD = getpass("Copernicus password: ")

copernicusmarine.login(
    username=USERNAME,
    password=PASSWORD
)

print("\nCopernicus authentication successful.")


# ============================================================
# 6. CONFIGURATION
# ============================================================

START_YEAR = 2000
END_YEAR   = 2005


# ------------------------------------------------------------
# Region
# ------------------------------------------------------------

MIN_LON = 45
MAX_LON = 105

MIN_LAT = 5
MAX_LAT = 30


# ------------------------------------------------------------
# GLORYS depth
# ------------------------------------------------------------

GLORYS_DEPTH_MIN = 0
GLORYS_DEPTH_MAX = 1200


# ============================================================
# 7. DATASET IDs
# ============================================================

GLORYS_DATASET = (
    "cmems_mod_glo_phy_my_0.083deg_P1D-m"
)

SST_DATASET = (
    "METOFFICE-GLO-SST-L4-REP-OBS-SST"
)

SSS_DATASET = (
    "cmems_obs-mob_glo_phy-sss_my_multi_P1D"
)

SLA_DATASET = (
    "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D"
)

CURRENT_DATASET = (
    "cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m"
)


# IMPORTANT:
# Do NOT use WIND_GLO_PHY_L3_MY_012_005 as dataset_id.
# We use the actual ASCENDING / DESCENDING datasets.

WIND_ASC_DATASET = (
    "cmems_obs-wind_glo_phy_my_l3-quikscat-seawinds-asc-0.25deg_P1D-i"
)

WIND_DES_DATASET = (
    "cmems_obs-wind_glo_phy_my_l3-quikscat-seawinds-des-0.25deg_P1D-i"
)


# ============================================================
# 8. PRINT CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("DOWNLOAD CONFIGURATION")
print("=" * 70)

print(f"Years      : {START_YEAR} -> {END_YEAR}")
print(f"Latitude   : {MIN_LAT} -> {MAX_LAT}")
print(f"Longitude  : {MIN_LON} -> {MAX_LON}")
print(
    f"GLORYS depth: "
    f"{GLORYS_DEPTH_MIN} -> {GLORYS_DEPTH_MAX} m"
)

print("=" * 70)


# ============================================================
# 9. GENERIC DATASET DOWNLOAD FUNCTION
# ============================================================

def download_dataset_year(
    dataset_name,
    dataset_id,
    variables,
    folder,
    filename,
    year
):

    output_dir = f"{RAW}/{folder}"

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    outfile = (
        f"{output_dir}/{filename}_{year}.nc"
    )


    # --------------------------------------------------------
    # Skip if already exists
    # --------------------------------------------------------

    if os.path.isfile(outfile):

        size_gb = (
            os.path.getsize(outfile)
            / (1024 ** 3)
        )

        print(
            f"SKIP | {dataset_name} {year} "
            f"| {size_gb:.2f} GB"
        )

        return True


    # --------------------------------------------------------
    # Start download
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print(
        f"DOWNLOADING {dataset_name} "
        f"| YEAR {year}"
    )
    print("=" * 75)


    try:

        copernicusmarine.subset(

            dataset_id=dataset_id,

            variables=variables,

            minimum_longitude=MIN_LON,
            maximum_longitude=MAX_LON,

            minimum_latitude=MIN_LAT,
            maximum_latitude=MAX_LAT,

            start_datetime=(
                f"{year}-01-01T00:00:00"
            ),

            end_datetime=(
                f"{year}-12-31T23:59:59"
            ),

            output_directory=output_dir,

            output_filename=(
                f"{filename}_{year}.nc"
            )
        )


        # ----------------------------------------------------
        # Verify output
        # ----------------------------------------------------

        if os.path.isfile(outfile):

            size_gb = (
                os.path.getsize(outfile)
                / (1024 ** 3)
            )

            print(
                f"FINISHED | {dataset_name} {year} "
                f"| {size_gb:.2f} GB"
            )

            return True


        else:

            print(
                f"WARNING | {dataset_name} {year}: "
                f"file not found after download"
            )

            return False


    except Exception as e:

        print(
            f"ERROR | {dataset_name} {year}"
        )

        print(e)

        return False


# ============================================================
# 10. GLORYS DOWNLOAD FUNCTION
# ============================================================

def download_glorys_year(year):

    folder = f"{RAW}/GLORYS"

    os.makedirs(
        folder,
        exist_ok=True
    )

    outfile = (
        f"{folder}/GLORYS_thetao_{year}.nc"
    )


    # --------------------------------------------------------
    # Skip existing file
    # --------------------------------------------------------

    if os.path.isfile(outfile):

        size_gb = (
            os.path.getsize(outfile)
            / (1024 ** 3)
        )

        print(
            f"SKIP | GLORYS {year} "
            f"| {size_gb:.2f} GB"
        )

        return True


    print("\n" + "=" * 75)
    print(
        f"DOWNLOADING GLORYS "
        f"| YEAR {year}"
    )
    print("=" * 75)


    try:

        copernicusmarine.subset(

            dataset_id=GLORYS_DATASET,

            variables=[
                "thetao"
            ],

            minimum_longitude=MIN_LON,
            maximum_longitude=MAX_LON,

            minimum_latitude=MIN_LAT,
            maximum_latitude=MAX_LAT,

            minimum_depth=GLORYS_DEPTH_MIN,
            maximum_depth=GLORYS_DEPTH_MAX,

            start_datetime=(
                f"{year}-01-01T00:00:00"
            ),

            end_datetime=(
                f"{year}-12-31T23:59:59"
            ),

            output_directory=folder,

            output_filename=(
                f"GLORYS_thetao_{year}.nc"
            )
        )


        if os.path.isfile(outfile):

            size_gb = (
                os.path.getsize(outfile)
                / (1024 ** 3)
            )

            print(
                f"FINISHED | GLORYS {year} "
                f"| {size_gb:.2f} GB"
            )

            return True


        print(
            f"WARNING | GLORYS {year}: "
            f"file not found"
        )

        return False


    except Exception as e:

        print(
            f"ERROR | GLORYS {year}"
        )

        print(e)

        return False


# ============================================================
# 11. WIND ASC / DES FUNCTION
# ============================================================

def download_wind_pass(
    pass_name,
    dataset_id,
    year
):

    folder = f"{RAW}/WIND"

    os.makedirs(
        folder,
        exist_ok=True
    )

    outfile = (
        f"{folder}/WIND_{pass_name}_{year}.nc"
    )


    # --------------------------------------------------------
    # Skip existing
    # --------------------------------------------------------

    if os.path.isfile(outfile):

        size_gb = (
            os.path.getsize(outfile)
            / (1024 ** 3)
        )

        print(
            f"SKIP | WIND {pass_name} {year} "
            f"| {size_gb:.2f} GB"
        )

        return True


    print("\n" + "=" * 75)
    print(
        f"DOWNLOADING WIND {pass_name} "
        f"| YEAR {year}"
    )
    print("=" * 75)


    try:

        copernicusmarine.subset(

            dataset_id=dataset_id,

            variables=[
                "eastward_wind",
                "northward_wind"
            ],

            minimum_longitude=MIN_LON,
            maximum_longitude=MAX_LON,

            minimum_latitude=MIN_LAT,
            maximum_latitude=MAX_LAT,

            start_datetime=(
                f"{year}-01-01T00:00:00"
            ),

            end_datetime=(
                f"{year}-12-31T23:59:59"
            ),

            output_directory=folder,

            output_filename=(
                f"WIND_{pass_name}_{year}.nc"
            )
        )


        if os.path.isfile(outfile):

            size_gb = (
                os.path.getsize(outfile)
                / (1024 ** 3)
            )

            print(
                f"FINISHED | WIND {pass_name} {year} "
                f"| {size_gb:.2f} GB"
            )

            return True


        print(
            f"WARNING | WIND {pass_name} {year}: "
            f"file not found"
        )

        return False


    except Exception as e:

        print(
            f"ERROR | WIND {pass_name} {year}"
        )

        print(e)

        return False


# ============================================================
# 12. DOWNLOAD GLORYS
# ============================================================

print("\n")
print("#" * 80)
print("# GLORYS DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    download_glorys_year(year)


# ============================================================
# 13. DOWNLOAD SST
# ============================================================

print("\n")
print("#" * 80)
print("# SST DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    download_dataset_year(

        dataset_name="SST",

        dataset_id=SST_DATASET,

        variables=[
            "analysed_sst"
        ],

        folder="SST",

        filename="SST",

        year=year
    )


# ============================================================
# 14. DOWNLOAD SSS
# ============================================================

print("\n")
print("#" * 80)
print("# SSS DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    download_dataset_year(

        dataset_name="SSS",

        dataset_id=SSS_DATASET,

        variables=[
            "sos"
        ],

        folder="SSS",

        filename="SSS",

        year=year
    )


# ============================================================
# 15. DOWNLOAD SLA
# ============================================================

print("\n")
print("#" * 80)
print("# SLA DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    download_dataset_year(

        dataset_name="SLA",

        dataset_id=SLA_DATASET,

        variables=[
            "sla"
        ],

        folder="SLA",

        filename="SLA",

        year=year
    )


# ============================================================
# 16. DOWNLOAD SURFACE CURRENTS
# ============================================================

print("\n")
print("#" * 80)
print("# SURFACE CURRENT DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    download_dataset_year(

        dataset_name="CURRENT",

        dataset_id=CURRENT_DATASET,

        variables=[
            "uo",
            "vo"
        ],

        folder="CURRENT",

        filename="CURRENT",

        year=year
    )


# ============================================================
# 17. DOWNLOAD WIND ASCENDING + DESCENDING
# ============================================================

print("\n")
print("#" * 80)
print("# WIND ASC / DES DOWNLOAD")
print("#" * 80)

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    # Ascending
    download_wind_pass(
        "ASC",
        WIND_ASC_DATASET,
        year
    )

    # Descending
    download_wind_pass(
        "DES",
        WIND_DES_DATASET,
        year
    )


# ============================================================
# 18. FINAL FILE CHECK
# ============================================================

print("\n")
print("#" * 80)
print("# FINAL DOWNLOAD CHECK")
print("#" * 80)


for dataset in [
    "GLORYS",
    "SST",
    "SSS",
    "SLA",
    "CURRENT",
    "WIND",
    "ARGO"
]:

    folder = f"{RAW}/{dataset}"

    print("\n")
    print(f"DATASET: {dataset}")
    print(f"FOLDER : {folder}")

    if not os.path.exists(folder):

        print("FOLDER MISSING")
        continue


    files = sorted(
        f
        for f in os.listdir(folder)
        if f.endswith(".nc")
    )


    if not files:

        print("(empty)")

    else:

        for file in files:

            path = os.path.join(
                folder,
                file
            )

            size_gb = (
                os.path.getsize(path)
                / (1024 ** 3)
            )

            print(
                f"  {file} "
                f"| {size_gb:.2f} GB"
            )


# ============================================================
# 19. EXPECTED FILE COUNT
# ============================================================

YEARS = list(
    range(
        START_YEAR,
        END_YEAR + 1
    )
)

print("\n")
print("=" * 80)
print("EXPECTED DOWNLOAD")
print("=" * 80)

print(
    f"Years: {YEARS}"
)

print(
    f"Number of years: {len(YEARS)}"
)

print(
    "\nGLORYS : 6 files"
)

print(
    "SST    : 6 files"
)

print(
    "SSS    : 6 files"
)

print(
    "SLA    : 6 files"
)

print(
    "CURRENT: 6 files"
)

print(
    "WIND   : 12 files "
    "(6 ASC + 6 DES)"
)

print(
    "\nTotal expected .nc files: 42"
)

print(
    "\nARGO: NOT downloaded yet."
)

print("=" * 80)

print("\nDOWNLOAD PROCESS COMPLETE.")

Mounted at /content/drive
OceanEmbed folders created
ROOT:
/content/drive/MyDrive/OceanEmbed

RAW folders:
  /content/drive/MyDrive/OceanEmbed/raw/GLORYS
  /content/drive/MyDrive/OceanEmbed/raw/SST
  /content/drive/MyDrive/OceanEmbed/raw/SSS
  /content/drive/MyDrive/OceanEmbed/raw/SLA
  /content/drive/MyDrive/OceanEmbed/raw/CURRENT
  /content/drive/MyDrive/OceanEmbed/raw/WIND
  /content/drive/MyDrive/OceanEmbed/raw/ARGO

COPERNICUS MARINE LOGIN


INFO - 2026-09-10T02:27:15Z - Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO:copernicusmarine:Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.



Copernicus authentication successful.

DOWNLOAD CONFIGURATION
Years      : 2000 -> 2005
Latitude   : 5 -> 30
Longitude  : 45 -> 105
GLORYS depth: 0 -> 1200 m


################################################################################
# GLORYS DOWNLOAD
################################################################################

DOWNLOADING GLORYS | YEAR 2000


INFO - 2026-09-10T02:27:18Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-10T02:27:18Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-10T02:27:18Z - Some of your subset selection [0.0, 1200.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]


  0%|          | [00:00<?]

...

FINISHED | WIND ASC 2005 | 0.03 GB

DOWNLOADING WIND DES | YEAR 2005


INFO - 2026-09-10T04:53:20Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-10T04:53:20Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-10T04:53:40Z - Total size of the download: 33.45 MB.
INFO:copernicusmarine:Total size of the download: 33.45 MB.


FINISHED | WIND DES 2005 | 0.03 GB


################################################################################
# FINAL DOWNLOAD CHECK
################################################################################


DATASET: GLORYS
FOLDER : /content/drive/MyDrive/OceanEmbed/raw/GLORYS
  GLORYS_thetao_2000.nc | 5.33 GB
  GLORYS_thetao_2001.nc | 5.31 GB
  GLORYS_thetao_2002.nc | 5.31 GB
  GLORYS_thetao_2003.nc | 5.31 GB
  GLORYS_thetao_2004.nc | 5.33 GB
  GLORYS_thetao_2005.nc | 5.31 GB


DATASET: SST
FOLDER : /content/drive/MyDrive/OceanEmbed/raw/SST
  SST_2000.nc | 0.41 GB
  SST_2001.nc | 0.41 GB
  SST_2002.nc | 0.41 GB
  SST_2003.nc | 0.41 GB
  SST_2004.nc | 0.41 GB
  SST_2005.nc | 0.41 GB


DATASET: SSS
FOLDER : /content/drive/MyDrive/OceanEmbed/raw/SSS
  SSS_2000.nc | 0.13 GB
  SSS_2001.nc | 0.13 GB
  SSS_2002.nc | 0.13 GB
  SSS_2003.nc | 0.13 GB
  SSS_2004.nc | 0.13 GB
  SSS_2005.nc | 0.13 GB


DATASET: SLA
FOLDER : /content/drive/MyDrive/OceanEmbed/raw/SLA
  SLA_2000.nc | 

In [ ]:
!pip install -q xarray netCDF4 dask

import os
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
RAW = f"{ROOT}/raw"
PROCESSED = f"{ROOT}/processed"

os.makedirs(PROCESSED, exist_ok=True)

print("ROOT      :", ROOT)
print("RAW       :", RAW)
print("PROCESSED :", PROCESSED)

ROOT      : /content/drive/MyDrive/OceanEmbed
RAW       : /content/drive/MyDrive/OceanEmbed/raw
PROCESSED : /content/drive/MyDrive/OceanEmbed/processed


In [ ]:
# ============================================================
# TARGET GRID
# ============================================================

TARGET_LAT = 5.125 + 0.25 * np.arange(100)
TARGET_LON = 45.125 + 0.25 * np.arange(240)

TARGET_DEPTHS = np.array([
    0, 5, 10, 20, 30,
    50, 75, 100, 125, 150,
    200, 300, 500, 700, 1000
])

print("Target latitude :", TARGET_LAT[0], "to", TARGET_LAT[-1])
print("Target longitude:", TARGET_LON[0], "to", TARGET_LON[-1])
print("Target depths   :", TARGET_DEPTHS)
print("Grid size       :", len(TARGET_LAT), "x", len(TARGET_LON))

Target latitude : 5.125 to 29.875
Target longitude: 45.125 to 104.875
Target depths   : [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]
Grid size       : 100 x 240


In [ ]:
# ============================================================
# REGRID TO 0.25 DEG
# ============================================================

def regrid_to_target(ds):
    return ds.interp(
        latitude=xr.DataArray(TARGET_LAT, dims="latitude"),
        longitude=xr.DataArray(TARGET_LON, dims="longitude"),
        method="linear"
    )

In [ ]:
# ============================================================
# GLORYS PROCESSING
# 2000-2005
# thetao -> 15 target depths
# ============================================================

glorys_list = []

for year in range(2000, 2006):

    path = f"{RAW}/GLORYS/GLORYS_thetao_{year}.nc"

    print(f"\nOpening GLORYS {year}")
    print(path)

    ds = xr.open_dataset(
        path,
        chunks={"time": 30}
    )

    print(ds)

    # Keep only thetao
    thetao = ds["thetao"]

    # --------------------------------------------------------
    # Vertical interpolation
    # --------------------------------------------------------

    thetao = thetao.interp(
        depth=TARGET_DEPTHS,
        method="linear"
    )

    # --------------------------------------------------------
    # Spatial interpolation
    # --------------------------------------------------------

    thetao = thetao.interp(
        latitude=xr.DataArray(
            TARGET_LAT,
            dims="latitude"
        ),
        longitude=xr.DataArray(
            TARGET_LON,
            dims="longitude"
        ),
        method="linear"
    )

    thetao = thetao.astype("float32")

    glorys_list.append(thetao)

    ds.close()

    print(f"GLORYS {year} processed.")


# ------------------------------------------------------------
# Combine all years
# ------------------------------------------------------------

target_thetao = xr.concat(
    glorys_list,
    dim="time"
)

target_thetao = target_thetao.sortby("time")

target = target_thetao.to_dataset(
    name="thetao"
)

target.attrs["description"] = (
    "GLORYS subsurface ocean temperature "
    "interpolated to 15 standard depths "
    "on 0.25 degree North Indian Ocean grid"
)

target.attrs["target_depths_m"] = str(
    TARGET_DEPTHS.tolist()
)

print("\nFINAL TARGET:")
print(target)


Opening GLORYS 2000
/content/drive/MyDrive/OceanEmbed/raw/GLORYS/GLORYS_thetao_2000.nc
<xarray.Dataset> Size: 23GB
Dimensions:    (time: 366, depth: 36, latitude: 301, longitude: 721)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2000-01-01 2000-01-02 ... 2000-12-31
  * depth      (depth) float32 144B 0.494 1.541 2.646 ... 763.3 902.3 1.062e+03
  * latitude   (latitude) float32 1kB 5.0 5.083 5.167 5.25 ... 29.83 29.92 30.0
  * longitude  (longitude) float32 3kB 45.0 45.08 45.17 ... 104.8 104.9 105.0
Data variables:
    thetao     (time, depth, latitude, longitude) float64 23GB dask.array<chunksize=(30, 36, 301, 721), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://ww

In [ ]:
# shifting from cpu to gpu dude to crash


TARGET_PATH = f"{PROCESSED}/target.nc"

print("\nSaving target...")

target.to_netcdf(
    TARGET_PATH,
    engine="netcdf4"
)

print("Saved:")
print(TARGET_PATH)

print(
    "Size:",
    os.path.getsize(TARGET_PATH) / (1024**3),
    "GB"
)

NameError: name 'PROCESSED' is not defined

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/OceanEmbed"
RAW = f"{ROOT}/raw"
PROCESSED = f"{ROOT}/processed"

print("ROOT:", ROOT)
print("RAW:", RAW)
print("PROCESSED:", PROCESSED)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/OceanEmbed
RAW: /content/drive/MyDrive/OceanEmbed/raw
PROCESSED: /content/drive/MyDrive/OceanEmbed/processed


In [ ]:
import os

for folder in ["GLORYS", "SST", "SSS", "SLA", "CURRENT", "WIND"]:
    path = f"{RAW}/{folder}"

    files = sorted(
        f for f in os.listdir(path)
        if f.endswith(".nc")
    )

    print(f"\n{folder}: {len(files)} files")

    for f in files:
        size = os.path.getsize(
            os.path.join(path, f)
        ) / (1024**3)

        print(f"  {f} -> {size:.2f} GB")


GLORYS: 6 files
  GLORYS_thetao_2000.nc -> 5.33 GB
  GLORYS_thetao_2001.nc -> 5.31 GB
  GLORYS_thetao_2002.nc -> 5.31 GB
  GLORYS_thetao_2003.nc -> 5.31 GB
  GLORYS_thetao_2004.nc -> 5.33 GB
  GLORYS_thetao_2005.nc -> 5.31 GB

SST: 6 files
  SST_2000.nc -> 0.41 GB
  SST_2001.nc -> 0.41 GB
  SST_2002.nc -> 0.41 GB
  SST_2003.nc -> 0.41 GB
  SST_2004.nc -> 0.41 GB
  SST_2005.nc -> 0.41 GB

SSS: 6 files
  SSS_2000.nc -> 0.13 GB
  SSS_2001.nc -> 0.13 GB
  SSS_2002.nc -> 0.13 GB
  SSS_2003.nc -> 0.13 GB
  SSS_2004.nc -> 0.13 GB
  SSS_2005.nc -> 0.13 GB

SLA: 6 files
  SLA_2000.nc -> 0.13 GB
  SLA_2001.nc -> 0.13 GB
  SLA_2002.nc -> 0.13 GB
  SLA_2003.nc -> 0.13 GB
  SLA_2004.nc -> 0.13 GB
  SLA_2005.nc -> 0.13 GB

CURRENT: 6 files
  CURRENT_2000.nc -> 0.07 GB
  CURRENT_2001.nc -> 0.07 GB
  CURRENT_2002.nc -> 0.07 GB
  CURRENT_2003.nc -> 0.07 GB
  CURRENT_2004.nc -> 0.07 GB
  CURRENT_2005.nc -> 0.07 GB

WIND: 12 files
  WIND_ASC_2000.nc -> 0.03 GB
  WIND_ASC_2001.nc -> 0.03 GB
  WIND_ASC_20

In [ ]:
!pip install -q netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 1: RAM-SAFE GLORYS PREPROCESSING
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
RAW = f"{ROOT}/raw"
PROCESSED = f"{ROOT}/processed"

os.makedirs(PROCESSED, exist_ok=True)

# ------------------------------------------------------------
# Target 0.25° grid
# ------------------------------------------------------------

TARGET_LAT = 5.125 + 0.25 * np.arange(100)
TARGET_LON = 45.125 + 0.25 * np.arange(240)

# ------------------------------------------------------------
# Required 15 depths
# ------------------------------------------------------------

TARGET_DEPTHS = np.array([
    0, 5, 10, 20, 30,
    50, 75, 100, 125, 150,
    200, 300, 500, 700, 1000
])

print("Target grid:")
print("Latitude :", TARGET_LAT[0], "->", TARGET_LAT[-1])
print("Longitude:", TARGET_LON[0], "->", TARGET_LON[-1])

print("\nTarget depths:")
print(TARGET_DEPTHS)

# ============================================================
# PROCESS YEAR BY YEAR
# ============================================================

for year in range(2000, 2006):

    print("\n" + "=" * 80)
    print(f"GLORYS PROCESSING: {year}")
    print("=" * 80)

    input_file = (
        f"{RAW}/GLORYS/"
        f"GLORYS_thetao_{year}.nc"
    )

    output_file = (
        f"{PROCESSED}/"
        f"target_{year}.nc"
    )

    # --------------------------------------------------------
    # Skip if already successfully processed
    # --------------------------------------------------------

    if os.path.exists(output_file):

        print("Already exists:")
        print(output_file)

        continue

    # --------------------------------------------------------
    # Open with chunks
    # --------------------------------------------------------

    print("Opening:")
    print(input_file)

    ds = xr.open_dataset(
        input_file,
        chunks={
            "time": 10
        }
    )

    print("\nOriginal dimensions:")
    print(ds.dims)

    thetao = ds["thetao"]

    # --------------------------------------------------------
    # Vertical interpolation
    # --------------------------------------------------------

    print("\n1/2 Vertical interpolation...")

    thetao = thetao.interp(
        depth=TARGET_DEPTHS,
        method="linear"
    )

    print(
        "Depth dimension:",
        thetao.depth.values
    )

    # --------------------------------------------------------
    # Spatial interpolation
    # --------------------------------------------------------

    print("\n2/2 Spatial interpolation...")

    thetao = thetao.interp(
        latitude=xr.DataArray(
            TARGET_LAT,
            dims="latitude"
        ),
        longitude=xr.DataArray(
            TARGET_LON,
            dims="longitude"
        ),
        method="linear"
    )

    # --------------------------------------------------------
    # Float32 to reduce storage/RAM
    # --------------------------------------------------------

    thetao = thetao.astype("float32")

    target_year = thetao.to_dataset(
        name="thetao"
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    target_year.attrs["description"] = (
        "GLORYS thetao interpolated to "
        "15 standard depths and 0.25 degree grid"
    )

    target_year.attrs["target_depths_m"] = (
        str(TARGET_DEPTHS.tolist())
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    print("\nSaving:")
    print(output_file)

    target_year.to_netcdf(
        output_file,
        engine="netcdf4"
    )

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    if os.path.exists(output_file):

        size_gb = (
            os.path.getsize(output_file)
            / (1024 ** 3)
        )

        print(
            f"\nSUCCESS: {year}"
        )

        print(
            f"Output size: {size_gb:.2f} GB"
        )

    else:

        print(
            f"\nERROR: output file missing for {year}"
        )

    # --------------------------------------------------------
    # Free RAM
    # --------------------------------------------------------

    ds.close()

    del ds
    del thetao
    del target_year

    gc.collect()

    print(
        f"RAM cleaned after {year}"
    )


print("\n" + "=" * 80)
print("GLORYS PREPROCESSING COMPLETE")
print("=" * 80)

Target grid:
Latitude : 5.125 -> 29.875
Longitude: 45.125 -> 104.875

Target depths:
[   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]

GLORYS PROCESSING: 2000
Opening:
/content/drive/MyDrive/OceanEmbed/raw/GLORYS/GLORYS_thetao_2000.nc

Original dimensions:
FrozenMappingWarningOnValuesAccess({'time': 366, 'depth': 36, 'latitude': 301, 'longitude': 721})

1/2 Vertical interpolation...
Depth dimension: [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]

2/2 Spatial interpolation...

Saving:
/content/drive/MyDrive/OceanEmbed/processed/target_2000.nc

SUCCESS: 2000
Output size: 0.49 GB
RAM cleaned after 2000

GLORYS PROCESSING: 2001
Opening:
/content/drive/MyDrive/OceanEmbed/raw/GLORYS/GLORYS_thetao_2001.nc

Original dimensions:
FrozenMappingWarningOnValuesAccess({'time': 365, 'depth': 36, 'latitude': 301, 'longitude': 721})

1/2 Vertical interpolation...
Depth dimension: [   0    5   10   20   30   50   75  100  125  150  200  300  50

In [ ]:
# ============================================================
# OceanEmbed
# SURFACE DATA PREPROCESSING
#
# Inputs:
#   SST
#   SSS
#   SLA
#   Current U
#   Current V
#   Wind U
#   Wind V
#
# Output:
#   processed/surface_2000.nc
#   ...
#   processed/surface_2005.nc
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
RAW = f"{ROOT}/raw"
PROCESSED = f"{ROOT}/processed"

os.makedirs(PROCESSED, exist_ok=True)


# ============================================================
# TARGET GRID
# ============================================================

TARGET_LAT = 5.125 + 0.25 * np.arange(100)
TARGET_LON = 45.125 + 0.25 * np.arange(240)


# ============================================================
# HELPER: FIND DEPTH-0 VARIABLE
# ============================================================

def select_surface(da):

    if "depth" in da.dims:

        # Find nearest available surface level
        da = da.sel(depth=0, method="nearest")

    return da


# ============================================================
# HELPER: REGRID
# ============================================================

def regrid_to_target(da):

    return da.interp(
        latitude=xr.DataArray(TARGET_LAT, dims="latitude"),
        longitude=xr.DataArray(TARGET_LON, dims="longitude"),
        method="linear"
    )


# ============================================================
# PROCESS EACH YEAR
# ============================================================

for year in range(2000, 2006):

    print("\n" + "=" * 80)
    print(f"SURFACE PREPROCESSING: {year}")
    print("=" * 80)

    output_file = f"{PROCESSED}/surface_{year}.nc"

    if os.path.exists(output_file):
        print("Already exists:")
        print(output_file)
        continue


    # ========================================================
    # SST
    # ========================================================

    print("\n1/5 SST")

    sst_file = f"{RAW}/SST/SST_{year}.nc"

    ds_sst = xr.open_dataset(
        sst_file,
        chunks={"time": 30}
    )

    sst = ds_sst["analysed_sst"]

    # Kelvin -> Celsius if necessary
    if float(sst.mean().compute()) > 100:

        print("SST detected in Kelvin -> converting to Celsius")

        sst = sst - 273.15

    sst = regrid_to_target(sst)

    sst = sst.astype("float32")


    # ========================================================
    # SSS
    # ========================================================

    print("2/5 SSS")

    sss_file = f"{RAW}/SSS/SSS_{year}.nc"

    ds_sss = xr.open_dataset(
        sss_file,
        chunks={"time": 30}
    )

    sss = ds_sss["sos"]

    sss = select_surface(sss)

    sss = regrid_to_target(sss)

    sss = sss.astype("float32")


    # ========================================================
    # SLA
    # ========================================================

    print("3/5 SLA")

    sla_file = f"{RAW}/SLA/SLA_{year}.nc"

    ds_sla = xr.open_dataset(
        sla_file,
        chunks={"time": 30}
    )

    sla = ds_sla["sla"]

    sla = regrid_to_target(sla)

    sla = sla.astype("float32")


    # ========================================================
    # SURFACE CURRENTS
    # ========================================================

    print("4/5 SURFACE CURRENTS")

    current_file = f"{RAW}/CURRENT/CURRENT_{year}.nc"

    ds_cur = xr.open_dataset(
        current_file,
        chunks={"time": 30}
    )

    uo = select_surface(ds_cur["uo"])
    vo = select_surface(ds_cur["vo"])

    uo = regrid_to_target(uo)
    vo = regrid_to_target(vo)

    uo = uo.astype("float32")
    vo = vo.astype("float32")


    # ========================================================
    # WIND ASCENDING + DESCENDING
    # ========================================================

    print("5/5 WIND")

    wind_asc_file = f"{RAW}/WIND/WIND_ASC_{year}.nc"
    wind_des_file = f"{RAW}/WIND/WIND_DES_{year}.nc"

    ds_wind_asc = xr.open_dataset(
        wind_asc_file,
        chunks={"time": 30}
    )

    ds_wind_des = xr.open_dataset(
        wind_des_file,
        chunks={"time": 30}
    )


    wind_u_asc = ds_wind_asc["eastward_wind"]
    wind_v_asc = ds_wind_asc["northward_wind"]

    wind_u_des = ds_wind_des["eastward_wind"]
    wind_v_des = ds_wind_des["northward_wind"]


    # Regrid each pass
    wind_u_asc = regrid_to_target(wind_u_asc)
    wind_v_asc = regrid_to_target(wind_v_asc)

    wind_u_des = regrid_to_target(wind_u_des)
    wind_v_des = regrid_to_target(wind_v_des)


    # ========================================================
    # COMBINE ASC + DES
    # ========================================================

    wind_u = xr.concat(
        [wind_u_asc, wind_u_des],
        dim="pass"
    ).mean(
        dim="pass",
        skipna=True
    )

    wind_v = xr.concat(
        [wind_v_asc, wind_v_des],
        dim="pass"
    ).mean(
        dim="pass",
        skipna=True
    )


    wind_u = wind_u.astype("float32")
    wind_v = wind_v.astype("float32")


    # ========================================================
    # ALIGN ALL DATASETS ON DAILY TIME
    # ========================================================

    print("\nAligning daily time...")

    surface = xr.Dataset({

        "sst": sst,
        "sss": sss,
        "sla": sla,
        "uo": uo,
        "vo": vo,
        "wind_u": wind_u,
        "wind_v": wind_v

    }).sortby("time")


    # ========================================================
    # FORCE COMMON TIME
    # ========================================================

    # Remove duplicate timestamps if any
    _, index = np.unique(
        surface.time.values,
        return_index=True
    )

    surface = surface.isel(
        time=np.sort(index)
    )


    # ========================================================
    # CHUNK FOR SAFE STORAGE
    # ========================================================

    surface = surface.chunk({
        "time": 10,
        "latitude": 25,
        "longitude": 60
    })


    # ========================================================
    # ATTRIBUTES
    # ========================================================

    surface.attrs["description"] = (
        "OceanEmbed daily surface inputs on "
        "0.25 degree North Indian Ocean grid"
    )

    surface.attrs["variables"] = (
        "SST, SSS, SLA, surface U/V current, "
        "surface U/V wind"
    )


    # ========================================================
    # SAVE
    # ========================================================

    print("\nSaving:")
    print(output_file)

    encoding = {}

    for var in surface.data_vars:

        encoding[var] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }


    surface.to_netcdf(
        output_file,
        engine="netcdf4",
        encoding=encoding
    )


    # ========================================================
    # CLEAN RAM
    # ========================================================

    ds_sst.close()
    ds_sss.close()
    ds_sla.close()
    ds_cur.close()
    ds_wind_asc.close()
    ds_wind_des.close()

    del (
        ds_sst,
        ds_sss,
        ds_sla,
        ds_cur,
        ds_wind_asc,
        ds_wind_des,
        sst,
        sss,
        sla,
        uo,
        vo,
        wind_u,
        wind_v,
        surface
    )

    gc.collect()

    print(f"\nSUCCESS: surface {year}")


SURFACE PREPROCESSING: 2000

1/5 SST
SST detected in Kelvin -> converting to Celsius
2/5 SSS
3/5 SLA
4/5 SURFACE CURRENTS
5/5 WIND

Aligning daily time...

Saving:
/content/drive/MyDrive/OceanEmbed/processed/surface_2000.nc

SUCCESS: surface 2000

SURFACE PREPROCESSING: 2001

1/5 SST
SST detected in Kelvin -> converting to Celsius
2/5 SSS
3/5 SLA
4/5 SURFACE CURRENTS
5/5 WIND

Aligning daily time...

Saving:
/content/drive/MyDrive/OceanEmbed/processed/surface_2001.nc

SUCCESS: surface 2001

SURFACE PREPROCESSING: 2002

1/5 SST
SST detected in Kelvin -> converting to Celsius
2/5 SSS
3/5 SLA
4/5 SURFACE CURRENTS
5/5 WIND

Aligning daily time...

Saving:
/content/drive/MyDrive/OceanEmbed/processed/surface_2002.nc

SUCCESS: surface 2002

SURFACE PREPROCESSING: 2003

1/5 SST
SST detected in Kelvin -> converting to Celsius
2/5 SSS
3/5 SLA
4/5 SURFACE CURRENTS
5/5 WIND

Aligning daily time...

Saving:
/content/drive/MyDrive/OceanEmbed/processed/surface_2003.nc

SUCCESS: surface 2003

SURFACE

In [ ]:
# ============================================================
# STEP 2: SURFACE DATA PREPROCESSING
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
RAW = f"{ROOT}/raw"
PROCESSED = f"{ROOT}/processed"

# ------------------------------------------------------------
# TARGET GRID
# ------------------------------------------------------------

TARGET_LAT = 5.125 + 0.25 * np.arange(100)
TARGET_LON = 45.125 + 0.25 * np.arange(240)


# ------------------------------------------------------------
# REGRID FUNCTION
# ------------------------------------------------------------

def regrid(da):

    return da.interp(
        latitude=xr.DataArray(TARGET_LAT, dims="latitude"),
        longitude=xr.DataArray(TARGET_LON, dims="longitude"),
        method="linear"
    )


# ------------------------------------------------------------
# SELECT SURFACE LEVEL IF DEPTH EXISTS
# ------------------------------------------------------------

def surface_level(da):

    if "depth" in da.dims:
        da = da.sel(depth=0, method="nearest")

    return da


# ============================================================
# PROCESS YEAR BY YEAR
# ============================================================

for year in range(2000, 2006):

    print("\n" + "=" * 80)
    print(f"SURFACE PREPROCESSING: {year}")
    print("=" * 80)

    output_file = f"{PROCESSED}/surface_{year}.nc"

    if os.path.exists(output_file):

        print("Already exists:")
        print(output_file)
        continue


    # ========================================================
    # 1. SST
    # ========================================================

    print("\n[1/5] SST")

    ds_sst = xr.open_dataset(
        f"{RAW}/SST/SST_{year}.nc",
        chunks={"time": 10}
    )

    sst = ds_sst["analysed_sst"]

    # Check Kelvin/Celsius
    mean_sst = float(sst.mean().compute())

    if mean_sst > 100:

        print("SST is Kelvin -> converting to Celsius")

        sst = sst - 273.15

    else:

        print("SST already Celsius")

    sst = regrid(sst).astype("float32")


    # ========================================================
    # 2. SSS
    # ========================================================

    print("[2/5] SSS")

    ds_sss = xr.open_dataset(
        f"{RAW}/SSS/SSS_{year}.nc",
        chunks={"time": 10}
    )

    sss = surface_level(
        ds_sss["sos"]
    )

    sss = regrid(sss).astype("float32")


    # ========================================================
    # 3. SLA
    # ========================================================

    print("[3/5] SLA")

    ds_sla = xr.open_dataset(
        f"{RAW}/SLA/SLA_{year}.nc",
        chunks={"time": 10}
    )

    sla = regrid(
        ds_sla["sla"]
    ).astype("float32")


    # ========================================================
    # 4. SURFACE CURRENT U/V
    # ========================================================

    print("[4/5] CURRENT U/V")

    ds_cur = xr.open_dataset(
        f"{RAW}/CURRENT/CURRENT_{year}.nc",
        chunks={"time": 10}
    )

    uo = surface_level(
        ds_cur["uo"]
    )

    vo = surface_level(
        ds_cur["vo"]
    )

    uo = regrid(uo).astype("float32")
    vo = regrid(vo).astype("float32")


    # ========================================================
    # 5. WIND ASCENDING + DESCENDING
    # ========================================================

    print("[5/5] WIND U/V")

    ds_asc = xr.open_dataset(
        f"{RAW}/WIND/WIND_ASC_{year}.nc",
        chunks={"time": 10}
    )

    ds_des = xr.open_dataset(
        f"{RAW}/WIND/WIND_DES_{year}.nc",
        chunks={"time": 10}
    )


    asc_u = regrid(
        ds_asc["eastward_wind"]
    )

    asc_v = regrid(
        ds_asc["northward_wind"]
    )

    des_u = regrid(
        ds_des["eastward_wind"]
    )

    des_v = regrid(
        ds_des["northward_wind"]
    )


    # --------------------------------------------------------
    # Combine ascending + descending
    # --------------------------------------------------------

    wind_u = xr.concat(
        [asc_u, des_u],
        dim="pass"
    ).mean(
        dim="pass",
        skipna=True
    )

    wind_v = xr.concat(
        [asc_v, des_v],
        dim="pass"
    ).mean(
        dim="pass",
        skipna=True
    )


    wind_u = wind_u.astype("float32")
    wind_v = wind_v.astype("float32")


    # ========================================================
    # CREATE 7-CHANNEL DATASET
    # ========================================================

    surface = xr.Dataset({

        "sst": sst,
        "sss": sss,
        "sla": sla,
        "uo": uo,
        "vo": vo,
        "wind_u": wind_u,
        "wind_v": wind_v

    })


    # ========================================================
    # ALIGN TIME
    # ========================================================

    print("\nAligning time...")

    surface = surface.sortby("time")

    # Keep only unique timestamps
    _, unique_idx = np.unique(
        surface.time.values,
        return_index=True
    )

    surface = surface.isel(
        time=np.sort(unique_idx)
    )


    # ========================================================
    # CHUNK
    # ========================================================

    surface = surface.chunk({
        "time": 10,
        "latitude": 25,
        "longitude": 60
    })


    # ========================================================
    # METADATA
    # ========================================================

    surface.attrs["description"] = (
        "OceanEmbed surface input variables "
        "on 0.25 degree North Indian Ocean grid"
    )

    surface.attrs["channels"] = (
        "sst, sss, sla, uo, vo, wind_u, wind_v"
    )


    # ========================================================
    # SAVE
    # ========================================================

    print("\nSaving:")
    print(output_file)

    encoding = {}

    for variable in surface.data_vars:

        encoding[variable] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }

    surface.to_netcdf(
        output_file,
        engine="netcdf4",
        encoding=encoding
    )


    # ========================================================
    # VERIFY
    # ========================================================

    if os.path.exists(output_file):

        size_gb = (
            os.path.getsize(output_file)
            / (1024 ** 3)
        )

        print("\nSUCCESS:", year)
        print(f"Output size: {size_gb:.2f} GB")

    else:

        print("\nERROR: output missing!")


    # ========================================================
    # CLEAN RAM
    # ========================================================

    ds_sst.close()
    ds_sss.close()
    ds_sla.close()
    ds_cur.close()
    ds_asc.close()
    ds_des.close()

    del (
        ds_sst,
        ds_sss,
        ds_sla,
        ds_cur,
        ds_asc,
        ds_des,
        sst,
        sss,
        sla,
        uo,
        vo,
        asc_u,
        asc_v,
        des_u,
        des_v,
        wind_u,
        wind_v,
        surface
    )

    gc.collect()

    print(f"RAM cleaned after {year}")


print("\n" + "=" * 80)
print("SURFACE PREPROCESSING COMPLETE")
print("=" * 80)


SURFACE PREPROCESSING: 2000
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2000.nc

SURFACE PREPROCESSING: 2001
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2001.nc

SURFACE PREPROCESSING: 2002
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2002.nc

SURFACE PREPROCESSING: 2003
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2003.nc

SURFACE PREPROCESSING: 2004
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2004.nc

SURFACE PREPROCESSING: 2005
Already exists:
/content/drive/MyDrive/OceanEmbed/processed/surface_2005.nc

SURFACE PREPROCESSING COMPLETE


In [ ]:
# ============================================================
# STEP 2: VERIFY PROCESSED DATA
# ============================================================

import os
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = f"{ROOT}/processed"

YEARS = range(2000, 2006)

print("=" * 80)
print("OCEANEMBED - PROCESSED DATA VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# Expected configuration
# ------------------------------------------------------------

EXPECTED_SURFACE_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

EXPECTED_DEPTHS = np.array([
    0, 5, 10, 20, 30,
    50, 75, 100, 125, 150,
    200, 300, 500, 700, 1000
])

EXPECTED_LAT = 5.125 + 0.25 * np.arange(100)
EXPECTED_LON = 45.125 + 0.25 * np.arange(240)

all_good = True

# ============================================================
# 1. CHECK TARGET FILES
# ============================================================

print("\n" + "=" * 80)
print("1. CHECKING GLORYS TARGET FILES")
print("=" * 80)

for year in YEARS:

    path = f"{PROCESSED}/target_{year}.nc"

    print(f"\nTARGET {year}")
    print("-" * 50)

    if not os.path.exists(path):
        print("ERROR: File missing!")
        all_good = False
        continue

    ds = xr.open_dataset(path)

    print("File:", path)
    print("Dimensions:", dict(ds.sizes))
    print("Variables:", list(ds.data_vars))

    # ---- dimensions ----

    if ds.sizes.get("depth") != 15:
        print("ERROR: depth dimension is not 15")
        all_good = False

    if ds.sizes.get("latitude") != 100:
        print("ERROR: latitude dimension is not 100")
        all_good = False

    if ds.sizes.get("longitude") != 240:
        print("ERROR: longitude dimension is not 240")
        all_good = False

    # ---- depths ----

    depths = ds["depth"].values

    if not np.allclose(depths, EXPECTED_DEPTHS):
        print("ERROR: depth values do not match")
        print("Found:", depths)
        all_good = False
    else:
        print("Depths: OK")

    # ---- latitude ----

    lat = ds["latitude"].values

    if not np.allclose(lat, EXPECTED_LAT):
        print("ERROR: latitude grid mismatch")
        all_good = False
    else:
        print("Latitude grid: OK")

    # ---- longitude ----

    lon = ds["longitude"].values

    if not np.allclose(lon, EXPECTED_LON):
        print("ERROR: longitude grid mismatch")
        all_good = False
    else:
        print("Longitude grid: OK")

    # ---- variable ----

    if "thetao" not in ds:
        print("ERROR: thetao missing")
        all_good = False
    else:
        print("thetao: OK")

        # Check NaN percentage
        nan_fraction = float(
            ds["thetao"].isnull().mean().compute()
        )

        print(f"thetao NaN fraction: {nan_fraction:.4%}")

    print(
        "Time:",
        str(ds.time.values[0]),
        "->",
        str(ds.time.values[-1])
    )

    ds.close()


# ============================================================
# 2. CHECK SURFACE FILES
# ============================================================

print("\n" + "=" * 80)
print("2. CHECKING SURFACE INPUT FILES")
print("=" * 80)

for year in YEARS:

    path = f"{PROCESSED}/surface_{year}.nc"

    print(f"\nSURFACE {year}")
    print("-" * 50)

    if not os.path.exists(path):
        print("ERROR: File missing!")
        print("Run surface preprocessing first.")
        all_good = False
        continue

    ds = xr.open_dataset(path)

    print("File:", path)
    print("Dimensions:", dict(ds.sizes))
    print("Variables:", list(ds.data_vars))

    # ---- dimensions ----

    if ds.sizes.get("latitude") != 100:
        print("ERROR: latitude dimension mismatch")
        all_good = False

    if ds.sizes.get("longitude") != 240:
        print("ERROR: longitude dimension mismatch")
        all_good = False

    # ---- variables ----

    missing = [
        v for v in EXPECTED_SURFACE_VARS
        if v not in ds.data_vars
    ]

    if missing:
        print("ERROR: Missing variables:", missing)
        all_good = False
    else:
        print("All 7 surface variables: OK")

    # ---- grid ----

    if not np.allclose(
        ds["latitude"].values,
        EXPECTED_LAT
    ):
        print("ERROR: latitude grid mismatch")
        all_good = False
    else:
        print("Latitude grid: OK")

    if not np.allclose(
        ds["longitude"].values,
        EXPECTED_LON
    ):
        print("ERROR: longitude grid mismatch")
        all_good = False
    else:
        print("Longitude grid: OK")

    print(
        "Time:",
        str(ds.time.values[0]),
        "->",
        str(ds.time.values[-1])
    )

    # ---- NaN check for each variable ----

    for var in EXPECTED_SURFACE_VARS:

        if var in ds:

            nan_fraction = float(
                ds[var].isnull().mean().compute()
            )

            print(
                f"{var:8s} NaN: "
                f"{nan_fraction:.4%}"
            )

    ds.close()


# ============================================================
# 3. FINAL RESULT
# ============================================================

print("\n" + "=" * 80)

if all_good:
    print("ALL BASIC CHECKS PASSED")
    print("=" * 80)
    print()
    print("NEXT:")
    print("1. Temporal alignment")
    print("2. Train/Test split")
    print("3. Train-only normalization")
    print("4. Sequence creation")
    print("5. GPU model training")

else:
    print("SOME CHECKS FAILED")
    print("=" * 80)
    print("DO NOT TRAIN THE MODEL YET.")
    print("Send me the output and I will fix the issue.")

OCEANEMBED - PROCESSED DATA VERIFICATION

1. CHECKING GLORYS TARGET FILES

TARGET 2000
--------------------------------------------------
File: /content/drive/MyDrive/OceanEmbed/processed/target_2000.nc
Dimensions: {'time': 366, 'depth': 15, 'latitude': 100, 'longitude': 240}
Variables: ['thetao']
Depths: OK
Latitude grid: OK
Longitude grid: OK
thetao: OK
thetao NaN fraction: 61.1664%
Time: 2000-01-01T00:00:00.000000000 -> 2000-12-31T00:00:00.000000000

TARGET 2001
--------------------------------------------------
File: /content/drive/MyDrive/OceanEmbed/processed/target_2001.nc
Dimensions: {'time': 365, 'depth': 15, 'latitude': 100, 'longitude': 240}
Variables: ['thetao']
Depths: OK
Latitude grid: OK
Longitude grid: OK
thetao: OK
thetao NaN fraction: 61.1664%
Time: 2001-01-01T00:00:00.000000000 -> 2001-12-31T00:00:00.000000000

TARGET 2002
--------------------------------------------------
File: /content/drive/MyDrive/OceanEmbed/processed/target_2002.nc
Dimensions: {'time': 365, 'dept

In [ ]:
if os.path.exists(output_file):
    print("Already exists:")
    print(output_file)
    continue

SyntaxError: 'continue' not properly in loop (117405604.py, line 4)

In [ ]:
import os
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = f"{ROOT}/processed"

for year in range(2000, 2006):

    surface_file = f"{PROCESSED}/surface_{year}.nc"
    target_file = f"{PROCESSED}/target_{year}.nc"

    print("\n" + "=" * 60)
    print(f"YEAR: {year}")
    print("=" * 60)

    # Surface
    ds_s = xr.open_dataset(surface_file)

    print("SURFACE:")
    print("  dimensions:", dict(ds_s.sizes))
    print("  variables:", list(ds_s.data_vars))
    print("  time:", ds_s.time.values[0], "to", ds_s.time.values[-1])

    # Target
    ds_t = xr.open_dataset(target_file)

    print("\nTARGET:")
    print("  dimensions:", dict(ds_t.sizes))
    print("  variables:", list(ds_t.data_vars))
    print("  depths:", ds_t.depth.values)
    print("  time:", ds_t.time.values[0], "to", ds_t.time.values[-1])

    ds_s.close()
    ds_t.close()

print("\n" + "=" * 60)
print("VERIFICATION COMPLETE")
print("=" * 60)


YEAR: 2000
SURFACE:
  dimensions: {'time': 366, 'latitude': 100, 'longitude': 240}
  variables: ['sst', 'sss', 'sla', 'uo', 'vo', 'wind_u', 'wind_v']
  time: 2000-01-01T00:00:00.000000000 to 2000-12-31T00:00:00.000000000

TARGET:
  dimensions: {'time': 366, 'depth': 15, 'latitude': 100, 'longitude': 240}
  variables: ['thetao']
  depths: [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]
  time: 2000-01-01T00:00:00.000000000 to 2000-12-31T00:00:00.000000000

YEAR: 2001
SURFACE:
  dimensions: {'time': 365, 'latitude': 100, 'longitude': 240}
  variables: ['sst', 'sss', 'sla', 'uo', 'vo', 'wind_u', 'wind_v']
  time: 2001-01-01T00:00:00.000000000 to 2001-12-31T00:00:00.000000000

TARGET:
  dimensions: {'time': 365, 'depth': 15, 'latitude': 100, 'longitude': 240}
  variables: ['thetao']
  depths: [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]
  time: 2001-01-01T00:00:00.000000000 to 2001-12-31T00:00:00.000000000

YEAR: 2002
SURFACE:

In [ ]:
# temporal alignment.
# ============================================================
# OceanEmbed - STEP 3
# TEMPORAL ALIGNMENT
#
# INPUT:
#   processed/surface_YYYY.nc
#   processed/target_YYYY.nc
#
# OUTPUT:
#   processed/aligned_YYYY.nc
#
# TRAIN:
#   2000-2004
#
# TEST:
#   2005
# ============================================================

import os
import gc
import numpy as np
import xarray as xr


# ============================================================
# PATHS
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = f"{ROOT}/processed"

os.makedirs(PROCESSED, exist_ok=True)


# ============================================================
# EXPECTED VARIABLES
# ============================================================

SURFACE_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

TARGET_VAR = "thetao"


# ============================================================
# EXPECTED GRID
# ============================================================

TARGET_LAT = 5.125 + 0.25 * np.arange(100)
TARGET_LON = 45.125 + 0.25 * np.arange(240)

TARGET_DEPTHS = np.array([
    0, 5, 10, 20, 30,
    50, 75, 100, 125, 150,
    200, 300, 500, 700, 1000
])


# ============================================================
# PROCESS YEAR BY YEAR
# ============================================================

for year in range(2000, 2006):

    print("\n" + "=" * 80)
    print(f"TEMPORAL ALIGNMENT: {year}")
    print("=" * 80)

    surface_file = f"{PROCESSED}/surface_{year}.nc"
    target_file = f"{PROCESSED}/target_{year}.nc"
    output_file = f"{PROCESSED}/aligned_{year}.nc"


    # ========================================================
    # CHECK INPUT FILES
    # ========================================================

    if not os.path.exists(surface_file):
        print("ERROR: Surface file missing:")
        print(surface_file)
        continue

    if not os.path.exists(target_file):
        print("ERROR: Target file missing:")
        print(target_file)
        continue


    # ========================================================
    # SKIP IF ALREADY DONE
    # ========================================================

    if os.path.exists(output_file):

        size_gb = os.path.getsize(output_file) / (1024 ** 3)

        print("ALREADY EXISTS:")
        print(output_file)
        print(f"Size: {size_gb:.2f} GB")
        print("Skipping...")

        continue


    # ========================================================
    # OPEN FILES
    # ========================================================

    print("\nOpening surface:")
    print(surface_file)

    surface = xr.open_dataset(
        surface_file,
        chunks={
            "time": 10,
            "latitude": 25,
            "longitude": 60
        }
    )


    print("\nOpening target:")
    print(target_file)

    target = xr.open_dataset(
        target_file,
        chunks={
            "time": 10,
            "depth": 15,
            "latitude": 25,
            "longitude": 60
        }
    )


    # ========================================================
    # CHECK VARIABLES
    # ========================================================

    print("\nChecking variables...")

    missing_surface = [
        v for v in SURFACE_VARS
        if v not in surface.data_vars
    ]

    if missing_surface:
        print("ERROR: Missing surface variables:")
        print(missing_surface)

        surface.close()
        target.close()

        continue


    if TARGET_VAR not in target.data_vars:

        print("ERROR: thetao missing from target")

        surface.close()
        target.close()

        continue


    print("All required variables found.")


    # ========================================================
    # CHECK GRID
    # ========================================================

    print("\nChecking spatial grid...")

    lat_ok_surface = np.allclose(
        surface.latitude.values,
        TARGET_LAT
    )

    lat_ok_target = np.allclose(
        target.latitude.values,
        TARGET_LAT
    )

    lon_ok_surface = np.allclose(
        surface.longitude.values,
        TARGET_LON
    )

    lon_ok_target = np.allclose(
        target.longitude.values,
        TARGET_LON
    )

    depth_ok = np.allclose(
        target.depth.values,
        TARGET_DEPTHS
    )


    if not lat_ok_surface:
        raise ValueError(
            f"Surface latitude grid incorrect for {year}"
        )

    if not lat_ok_target:
        raise ValueError(
            f"Target latitude grid incorrect for {year}"
        )

    if not lon_ok_surface:
        raise ValueError(
            f"Surface longitude grid incorrect for {year}"
        )

    if not lon_ok_target:
        raise ValueError(
            f"Target longitude grid incorrect for {year}"
        )

    if not depth_ok:
        raise ValueError(
            f"Target depth grid incorrect for {year}"
        )


    print("Latitude grid: OK")
    print("Longitude grid: OK")
    print("Depth grid: OK")


    # ========================================================
    # SHOW ORIGINAL TIMES
    # ========================================================

    print("\nOriginal time ranges:")

    print(
        "Surface:",
        surface.time.values[0],
        "->",
        surface.time.values[-1]
    )

    print(
        "Target :",
        target.time.values[0],
        "->",
        target.time.values[-1]
    )


    # ========================================================
    # FIND COMMON DATES
    # ========================================================

    print("\nFinding common dates...")

    surface_times = surface.time.values
    target_times = target.time.values

    common_times = np.intersect1d(
        surface_times,
        target_times
    )

    print(
        "Surface days:",
        len(surface_times)
    )

    print(
        "Target days :",
        len(target_times)
    )

    print(
        "Common days :",
        len(common_times)
    )


    # ========================================================
    # CHECK COMMON DATES
    # ========================================================

    if len(common_times) == 0:

        print("ERROR: No common dates!")

        surface.close()
        target.close()

        continue


    # ========================================================
    # SELECT ONLY COMMON DATES
    # ========================================================

    print("\nSelecting common dates...")

    surface_aligned = surface.sel(
        time=common_times
    )

    target_aligned = target.sel(
        time=common_times
    )


    # ========================================================
    # FINAL TIME CHECK
    # ========================================================

    if not np.array_equal(
        surface_aligned.time.values,
        target_aligned.time.values
    ):

        raise ValueError(
            f"Surface and target times still don't match "
            f"for {year}"
        )


    print("Surface/target timestamps: EXACT MATCH")


    # ========================================================
    # BUILD ALIGNED DATASET
    # ========================================================

    aligned = xr.Dataset()

    for variable in SURFACE_VARS:

        aligned[variable] = surface_aligned[variable]


    aligned[TARGET_VAR] = target_aligned[TARGET_VAR]


    # ========================================================
    # CHUNK
    # ========================================================

    aligned = aligned.chunk({
        "time": 10,
        "depth": 15,
        "latitude": 25,
        "longitude": 60
    })


    # ========================================================
    # METADATA
    # ========================================================

    aligned.attrs["description"] = (
        "OceanEmbed temporally aligned surface inputs "
        "and GLORYS subsurface temperature target"
    )

    aligned.attrs["surface_variables"] = (
        "sst, sss, sla, uo, vo, wind_u, wind_v"
    )

    aligned.attrs["target_variable"] = "thetao"

    aligned.attrs["target_depths_m"] = str(
        TARGET_DEPTHS.tolist()
    )

    aligned.attrs["grid_resolution"] = "0.25 degree"

    aligned.attrs["region"] = (
        "5N-30N, 45E-105E"
    )


    # ========================================================
    # SAVE
    # ========================================================

    print("\nSaving:")
    print(output_file)

    encoding = {}

    for variable in aligned.data_vars:

        encoding[variable] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }


    aligned.to_netcdf(
        output_file,
        engine="netcdf4",
        encoding=encoding
    )


    # ========================================================
    # VERIFY OUTPUT
    # ========================================================

    if os.path.exists(output_file):

        size_gb = os.path.getsize(output_file) / (1024 ** 3)

        print("\nSUCCESS:", year)
        print(f"Aligned days: {len(common_times)}")
        print(f"Output size: {size_gb:.2f} GB")

    else:

        print("\nERROR: Output file was not created!")


    # ========================================================
    # CLEAN RAM
    # ========================================================

    surface.close()
    target.close()

    del (
        surface,
        target,
        surface_aligned,
        target_aligned,
        aligned,
        common_times
    )

    gc.collect()

    print(f"RAM cleaned after {year}")


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 80)
print("TEMPORAL ALIGNMENT COMPLETE")
print("=" * 80)

print("\nCreated:")
for year in range(2000, 2006):
    print(f"  aligned_{year}.nc")

print("\nNext:")
print("  TRAIN = 2000-2004")
print("  TEST  = 2005")
print("  Then  = train-only normalization")


TEMPORAL ALIGNMENT: 2000

Opening surface:
/content/drive/MyDrive/OceanEmbed/processed/surface_2000.nc

Opening target:
/content/drive/MyDrive/OceanEmbed/processed/target_2000.nc


/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(



Checking variables...
All required variables found.

Checking spatial grid...
Latitude grid: OK
Longitude grid: OK
Depth grid: OK

Original time ranges:
Surface: 2000-01-01T00:00:00.000000000 -> 2000-12-31T00:00:00.000000000
Target : 2000-01-01T00:00:00.000000000 -> 2000-12-31T00:00:00.000000000

Finding common dates...
Surface days: 366
Target days : 366
Common days : 366

Selecting common dates...
Surface/target timestamps: EXACT MATCH

Saving:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2000.nc

SUCCESS: 2000
Aligned days: 366
Output size: 0.23 GB
RAM cleaned after 2000

TEMPORAL ALIGNMENT: 2001

Opening surface:
/content/drive/MyDrive/OceanEmbed/processed/surface_2001.nc


/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(



Opening target:
/content/drive/MyDrive/OceanEmbed/processed/target_2001.nc

Checking variables...
All required variables found.

Checking spatial grid...
Latitude grid: OK
Longitude grid: OK
Depth grid: OK

Original time ranges:
Surface: 2001-01-01T00:00:00.000000000 -> 2001-12-31T00:00:00.000000000
Target : 2001-01-01T00:00:00.000000000 -> 2001-12-31T00:00:00.000000000

Finding common dates...
Surface days: 365
Target days : 365
Common days : 365

Selecting common dates...
Surface/target timestamps: EXACT MATCH

Saving:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2001.nc

SUCCESS: 2001
Aligned days: 365
Output size: 0.23 GB
RAM cleaned after 2001

TEMPORAL ALIGNMENT: 2002

Opening surface:
/content/drive/MyDrive/OceanEmbed/processed/surface_2002.nc


...


Opening target:
/content/drive/MyDrive/OceanEmbed/processed/target_2003.nc

Checking variables...
All required variables found.

Checking spatial grid...
Latitude grid: OK
Longitude grid: OK
Depth grid: OK

Original time ranges:
Surface: 2003-01-01T00:00:00.000000000 -> 2003-12-31T00:00:00.000000000
Target : 2003-01-01T00:00:00.000000000 -> 2003-12-31T00:00:00.000000000

Finding common dates...
Surface days: 365
Target days : 365
Common days : 365

Selecting common dates...
Surface/target timestamps: EXACT MATCH

Saving:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2003.nc

SUCCESS: 2003
Aligned days: 365
Output size: 0.23 GB
RAM cleaned after 2003

TEMPORAL ALIGNMENT: 2004

Opening surface:
/content/drive/MyDrive/OceanEmbed/processed/surface_2004.nc

Opening target:
/content/drive/MyDrive/OceanEmbed/processed/target_2004.nc


/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(



Checking variables...
All required variables found.

Checking spatial grid...
Latitude grid: OK
Longitude grid: OK
Depth grid: OK

Original time ranges:
Surface: 2004-01-01T00:00:00.000000000 -> 2004-12-31T00:00:00.000000000
Target : 2004-01-01T00:00:00.000000000 -> 2004-12-31T00:00:00.000000000

Finding common dates...
Surface days: 366
Target days : 366
Common days : 366

Selecting common dates...
Surface/target timestamps: EXACT MATCH

Saving:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2004.nc

SUCCESS: 2004
Aligned days: 366
Output size: 0.23 GB
RAM cleaned after 2004

TEMPORAL ALIGNMENT: 2005

Opening surface:
/content/drive/MyDrive/OceanEmbed/processed/surface_2005.nc


/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(
/tmp/ipykernel_4825/366744234.py:120: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  surface = xr.open_dataset(



Opening target:
/content/drive/MyDrive/OceanEmbed/processed/target_2005.nc

Checking variables...
All required variables found.

Checking spatial grid...
Latitude grid: OK
Longitude grid: OK
Depth grid: OK

Original time ranges:
Surface: 2005-01-01T00:00:00.000000000 -> 2005-12-31T00:00:00.000000000
Target : 2005-01-01T00:00:00.000000000 -> 2005-12-31T00:00:00.000000000

Finding common dates...
Surface days: 365
Target days : 365
Common days : 365

Selecting common dates...
Surface/target timestamps: EXACT MATCH

Saving:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2005.nc

SUCCESS: 2005
Aligned days: 365
Output size: 0.23 GB
RAM cleaned after 2005

TEMPORAL ALIGNMENT COMPLETE

Created:
  aligned_2000.nc
  aligned_2001.nc
  aligned_2002.nc
  aligned_2003.nc
  aligned_2004.nc
  aligned_2005.nc

Next:
  TRAIN = 2000-2004
  TEST  = 2005
  Then  = train-only normalization


In [ ]:
# ============================================================
# OceanEmbed - STEP 4
# TRAIN-ONLY NORMALIZATION STATISTICS
#
# TRAIN YEARS: 2000-2004
# TEST YEAR  : 2005
#
# IMPORTANT:
# 2005 is NEVER used to calculate mean/std
# ============================================================

import os
import gc
import json
import numpy as np
import xarray as xr


# ============================================================
# PATHS
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = f"{ROOT}/processed"

STATS_FILE = f"{PROCESSED}/train_normalization_stats.json"


# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_YEARS = range(2000, 2005)

SURFACE_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]


# ============================================================
# IF STATS ALREADY EXIST, DON'T RECOMPUTE
# ============================================================

if os.path.exists(STATS_FILE):

    print("=" * 80)
    print("TRAIN NORMALIZATION STATISTICS ALREADY EXIST")
    print("=" * 80)

    print(STATS_FILE)

    with open(STATS_FILE, "r") as f:
        stats = json.load(f)

    print("\nExisting statistics:")

    for variable in SURFACE_VARS:
        print(
            f"{variable:8s} "
            f"mean = {stats[variable]['mean']:.6f}   "
            f"std = {stats[variable]['std']:.6f}"
        )

    print("\nSkipping recalculation.")

else:

    print("=" * 80)
    print("CALCULATING TRAIN-ONLY NORMALIZATION STATISTICS")
    print("=" * 80)

    print("\nTraining years:")
    print(list(TRAIN_YEARS))

    print("\nTest year:")
    print("2005")

    print("\nIMPORTANT:")
    print("2005 is NOT used for mean/std calculation.")


    # ========================================================
    # ACCUMULATORS
    # ========================================================

    stats_accumulator = {}

    for variable in SURFACE_VARS:

        stats_accumulator[variable] = {
            "sum": 0.0,
            "sum_sq": 0.0,
            "count": 0
        }


    # ========================================================
    # PROCESS TRAIN YEARS ONE BY ONE
    # ========================================================

    for year in TRAIN_YEARS:

        print("\n" + "=" * 80)
        print(f"PROCESSING TRAIN YEAR: {year}")
        print("=" * 80)

        file_path = f"{PROCESSED}/aligned_{year}.nc"

        print("Opening:")
        print(file_path)

        ds = xr.open_dataset(
            file_path,
            chunks={
                "time": 10,
                "latitude": 25,
                "longitude": 60
            }
        )


        # ====================================================
        # EACH SURFACE VARIABLE
        # ====================================================

        for variable in SURFACE_VARS:

            print(f"\nCalculating statistics: {variable}")

            da = ds[variable]


            # ------------------------------------------------
            # Count valid values
            # ------------------------------------------------

            count = da.count().compute()

            count = int(count.values)


            # ------------------------------------------------
            # Sum
            # ------------------------------------------------

            total_sum = da.sum(
                skipna=True
            ).compute()

            total_sum = float(total_sum.values)


            # ------------------------------------------------
            # Sum of squares
            # ------------------------------------------------

            total_sum_sq = (
                (da ** 2)
                .sum(skipna=True)
                .compute()
            )

            total_sum_sq = float(total_sum_sq.values)


            # ------------------------------------------------
            # Accumulate
            # ------------------------------------------------

            stats_accumulator[variable]["sum"] += total_sum

            stats_accumulator[variable]["sum_sq"] += total_sum_sq

            stats_accumulator[variable]["count"] += count


            print(
                f"Valid values: {count:,}"
            )


        ds.close()

        del ds

        gc.collect()

        print(f"\nRAM cleaned after {year}")


    # ========================================================
    # CALCULATE FINAL MEAN / STD
    # ========================================================

    print("\n" + "=" * 80)
    print("FINAL TRAIN-ONLY STATISTICS")
    print("=" * 80)


    stats = {}


    for variable in SURFACE_VARS:

        total_sum = stats_accumulator[variable]["sum"]

        total_sum_sq = stats_accumulator[variable]["sum_sq"]

        count = stats_accumulator[variable]["count"]


        # ----------------------------------------------------
        # Mean
        # ----------------------------------------------------

        mean = total_sum / count


        # ----------------------------------------------------
        # Variance
        #
        # Var(X) = E[X²] - E[X]²
        # ----------------------------------------------------

        variance = (
            total_sum_sq / count
            - mean ** 2
        )


        # Numerical safety
        variance = max(variance, 0.0)


        std = np.sqrt(variance)


        stats[variable] = {
            "mean": float(mean),
            "std": float(std),
            "count": int(count)
        }


        print(
            f"{variable:8s} "
            f"mean = {mean:.6f}   "
            f"std = {std:.6f}   "
            f"count = {count:,}"
        )


    # ========================================================
    # SAVE STATISTICS
    # ========================================================

    print("\nSaving statistics:")

    with open(STATS_FILE, "w") as f:

        json.dump(
            stats,
            f,
            indent=4
        )


    print("\nSaved:")
    print(STATS_FILE)


    # ========================================================
    # FINAL CHECK
    # ========================================================

    print("\n" + "=" * 80)
    print("NORMALIZATION STATISTICS COMPLETE")
    print("=" * 80)

    print("\nThese statistics were calculated ONLY from:")
    print("2000-2004")

    print("\n2005 was NOT used.")

    print("\nSaved variables:")

    for variable in SURFACE_VARS:
        print(
            f"  {variable}: "
            f"mean={stats[variable]['mean']:.6f}, "
            f"std={stats[variable]['std']:.6f}"
        )

CALCULATING TRAIN-ONLY NORMALIZATION STATISTICS

Training years:
[2000, 2001, 2002, 2003, 2004]

Test year:
2005

IMPORTANT:
2005 is NOT used for mean/std calculation.

PROCESSING TRAIN YEAR: 2000
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2000.nc


/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(



Calculating statistics: sst
Valid values: 4,341,858

Calculating statistics: sss
Valid values: 4,277,808

Calculating statistics: sla
Valid values: 4,360,158

Calculating statistics: uo
Valid values: 4,202,412

Calculating statistics: vo
Valid values: 4,202,412

Calculating statistics: wind_u
Valid values: 3,446,677

Calculating statistics: wind_v
Valid values: 3,446,677

RAM cleaned after 2000

PROCESSING TRAIN YEAR: 2001
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2001.nc


/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(



Calculating statistics: sst
Valid values: 4,329,995

Calculating statistics: sss
Valid values: 4,266,120

Calculating statistics: sla
Valid values: 4,348,245

Calculating statistics: uo
Valid values: 4,190,722

Calculating statistics: vo
Valid values: 4,190,722

Calculating statistics: wind_u
Valid values: 3,424,159

Calculating statistics: wind_v
Valid values: 3,424,159

RAM cleaned after 2001

PROCESSING TRAIN YEAR: 2002
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2002.nc


...


Calculating statistics: sst
Valid values: 4,329,995

Calculating statistics: sss
Valid values: 4,266,120

Calculating statistics: sla
Valid values: 4,348,245

Calculating statistics: uo
Valid values: 4,190,930

Calculating statistics: vo
Valid values: 4,190,930

Calculating statistics: wind_u
Valid values: 3,445,909

Calculating statistics: wind_v
Valid values: 3,445,909

RAM cleaned after 2002

PROCESSING TRAIN YEAR: 2003
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2003.nc


/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(



Calculating statistics: sst
Valid values: 4,329,995

Calculating statistics: sss
Valid values: 4,266,120

Calculating statistics: sla
Valid values: 4,348,245

Calculating statistics: uo
Valid values: 4,190,930

Calculating statistics: vo
Valid values: 4,190,930

Calculating statistics: wind_u
Valid values: 3,459,925

Calculating statistics: wind_v
Valid values: 3,459,925

RAM cleaned after 2003

PROCESSING TRAIN YEAR: 2004
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2004.nc


/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_4825/349680688.py:118: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(



Calculating statistics: sst
Valid values: 4,341,858

Calculating statistics: sss
Valid values: 4,277,808

Calculating statistics: sla
Valid values: 4,360,158

Calculating statistics: uo
Valid values: 4,202,412

Calculating statistics: vo
Valid values: 4,202,412

Calculating statistics: wind_u
Valid values: 3,473,318

Calculating statistics: wind_v
Valid values: 3,473,318

RAM cleaned after 2004

FINAL TRAIN-ONLY STATISTICS
sst      mean = 28.022341   std = 1.788864   count = 21,673,701
sss      mean = 34.712441   std = 1.958337   count = 21,353,976
sla      mean = 0.013862   std = 0.093307   count = 21,765,051
uo       mean = 0.037250   std = 0.272300   count = 20,977,406
vo       mean = 0.009934   std = 0.223371   count = 20,977,406
wind_u   mean = 1.446099   std = 4.928651   count = 17,249,988
wind_v   mean = 0.171190   std = 4.440354   count = 17,249,988

Saving statistics:

Saved:
/content/drive/MyDrive/OceanEmbed/processed/train_normalization_stats.json

NORMALIZATION STATISTICS 

In [ ]:
# Start from New RUNTIME
import os
import json
import gc
import numpy as np
import xarray as xr

# ============================================================
# CONFIG
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

TRAIN_YEARS = [2000, 2001, 2002, 2003, 2004]
TEST_YEAR = 2005

TARGET_DEPTHS = np.array(
    [0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000],
    dtype=np.float32
)

OUTPUT_JSON = os.path.join(
    PROCESSED,
    "target_normalization_stats.json"
)

print("=" * 80)
print("CALCULATING TRAIN-ONLY TARGET NORMALIZATION STATISTICS")
print("=" * 80)

print("\nTraining years:")
print(TRAIN_YEARS)

print("\nTest year:")
print(TEST_YEAR)

print("\nIMPORTANT:")
print("2005 is NOT used for target mean/std calculation.")

# ============================================================
# ACCUMULATORS
# ============================================================

n_depths = len(TARGET_DEPTHS)

sum_theta = np.zeros(n_depths, dtype=np.float64)
sum_theta_sq = np.zeros(n_depths, dtype=np.float64)
count_theta = np.zeros(n_depths, dtype=np.int64)

# ============================================================
# PROCESS TRAIN YEARS
# ============================================================

for year in TRAIN_YEARS:

    path = os.path.join(
        PROCESSED,
        f"aligned_{year}.nc"
    )

    print("\n" + "=" * 80)
    print(f"PROCESSING TRAIN YEAR: {year}")
    print("=" * 80)

    print("Opening:")
    print(path)

    ds = xr.open_dataset(
        path,
        chunks={
            "time": 10,
            "latitude": 25,
            "longitude": 60
        }
    )

    thetao = ds["thetao"]

    print("thetao dimensions:", thetao.dims)
    print("thetao shape:", thetao.shape)

    # --------------------------------------------------------
    # Calculate depth-wise statistics
    # --------------------------------------------------------

    print("\nCalculating sums...")

    yearly_sum = thetao.sum(
        dim=("time", "latitude", "longitude"),
        skipna=True
    ).compute()

    print("Calculating squared sums...")

    yearly_sum_sq = (thetao ** 2).sum(
        dim=("time", "latitude", "longitude"),
        skipna=True
    ).compute()

    print("Calculating valid counts...")

    yearly_count = thetao.count(
        dim=("time", "latitude", "longitude")
    ).compute()

    # --------------------------------------------------------
    # Accumulate
    # --------------------------------------------------------

    sum_theta += yearly_sum.values.astype(np.float64)
    sum_theta_sq += yearly_sum_sq.values.astype(np.float64)
    count_theta += yearly_count.values.astype(np.int64)

    print("\nYear complete.")

    for i, depth in enumerate(TARGET_DEPTHS):
        print(
            f"Depth {depth:6.1f} m : "
            f"valid = {yearly_count.values[i]:,}"
        )

    ds.close()

    del ds, thetao
    del yearly_sum, yearly_sum_sq, yearly_count

    gc.collect()

    print(f"\nRAM cleaned after {year}")

# ============================================================
# FINAL MEAN / STD
# ============================================================

print("\n" + "=" * 80)
print("FINAL TRAIN-ONLY TARGET STATISTICS")
print("=" * 80)

mean_theta = sum_theta / count_theta

variance_theta = (
    sum_theta_sq / count_theta
    - mean_theta ** 2
)

# Numerical safety
variance_theta = np.maximum(variance_theta, 0.0)

std_theta = np.sqrt(variance_theta)

# ============================================================
# PRINT RESULTS
# ============================================================

for i, depth in enumerate(TARGET_DEPTHS):

    print(
        f"{depth:6.1f} m : "
        f"mean = {mean_theta[i]:.6f}   "
        f"std = {std_theta[i]:.6f}   "
        f"count = {count_theta[i]:,}"
    )

# ============================================================
# SAFETY CHECK
# ============================================================

if np.any(~np.isfinite(mean_theta)):
    raise ValueError("Some target means are invalid.")

if np.any(~np.isfinite(std_theta)):
    raise ValueError("Some target std values are invalid.")

if np.any(std_theta <= 0):
    raise ValueError("Some target std values are <= 0.")

# ============================================================
# SAVE JSON
# ============================================================

stats = {
    "training_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "depths": TARGET_DEPTHS.tolist(),
    "mean": mean_theta.tolist(),
    "std": std_theta.tolist(),
    "count": count_theta.tolist()
}

print("\nSaving statistics:")

with open(OUTPUT_JSON, "w") as f:
    json.dump(stats, f, indent=4)

print("Saved:")
print(OUTPUT_JSON)

print("\n" + "=" * 80)
print("TARGET NORMALIZATION STATISTICS COMPLETE")
print("=" * 80)

print("\n2000-2004 ONLY were used.")
print("2005 was NOT used.")

print("\nNext step:")
print("Apply surface normalization + prepare target normalization")
print("without filling target NaNs.")

CALCULATING TRAIN-ONLY TARGET NORMALIZATION STATISTICS

Training years:
[2000, 2001, 2002, 2003, 2004]

Test year:
2005

IMPORTANT:
2005 is NOT used for target mean/std calculation.

PROCESSING TRAIN YEAR: 2000
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2000.nc


/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


thetao dimensions: ('time', 'depth', 'latitude', 'longitude')
thetao shape: (366, 15, 100, 240)

Calculating sums...
Calculating squared sums...
Calculating valid counts...

Year complete.
Depth    0.0 m : valid = 0
Depth    5.0 m : valid = 4,290,252
Depth   10.0 m : valid = 4,163,616
Depth   20.0 m : valid = 4,049,058
Depth   30.0 m : valid = 3,930,474
Depth   50.0 m : valid = 3,765,408
Depth   75.0 m : valid = 3,610,224
Depth  100.0 m : valid = 3,528,240
Depth  125.0 m : valid = 3,508,110
Depth  150.0 m : valid = 3,493,104
Depth  200.0 m : valid = 3,458,700
Depth  300.0 m : valid = 3,431,616
Depth  500.0 m : valid = 3,373,788
Depth  700.0 m : valid = 3,319,986
Depth 1000.0 m : valid = 3,244,590

RAM cleaned after 2000

PROCESSING TRAIN YEAR: 2001
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2001.nc


/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


thetao dimensions: ('time', 'depth', 'latitude', 'longitude')
thetao shape: (365, 15, 100, 240)

Calculating sums...
Calculating squared sums...
Calculating valid counts...

Year complete.
Depth    0.0 m : valid = 0
Depth    5.0 m : valid = 4,278,530
Depth   10.0 m : valid = 4,152,240
Depth   20.0 m : valid = 4,037,995
Depth   30.0 m : valid = 3,919,735
Depth   50.0 m : valid = 3,755,120
Depth   75.0 m : valid = 3,600,360
Depth  100.0 m : valid = 3,518,600
Depth  125.0 m : valid = 3,498,525
Depth  150.0 m : valid = 3,483,560
Depth  200.0 m : valid = 3,449,250
Depth  300.0 m : valid = 3,422,240
Depth  500.0 m : valid = 3,364,570
Depth  700.0 m : valid = 3,310,915
Depth 1000.0 m : valid = 3,235,725

RAM cleaned after 2001

PROCESSING TRAIN YEAR: 2002
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2002.nc


...

thetao dimensions: ('time', 'depth', 'latitude', 'longitude')
thetao shape: (365, 15, 100, 240)

Calculating sums...
Calculating squared sums...
Calculating valid counts...

Year complete.
Depth    0.0 m : valid = 0
Depth    5.0 m : valid = 4,278,530
Depth   10.0 m : valid = 4,152,240
Depth   20.0 m : valid = 4,037,995
Depth   30.0 m : valid = 3,919,735
Depth   50.0 m : valid = 3,755,120
Depth   75.0 m : valid = 3,600,360
Depth  100.0 m : valid = 3,518,600
Depth  125.0 m : valid = 3,498,525
Depth  150.0 m : valid = 3,483,560
Depth  200.0 m : valid = 3,449,250
Depth  300.0 m : valid = 3,422,240
Depth  500.0 m : valid = 3,364,570
Depth  700.0 m : valid = 3,310,915
Depth 1000.0 m : valid = 3,235,725

RAM cleaned after 2003

PROCESSING TRAIN YEAR: 2004
Opening:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2004.nc


/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/2441475982.py:69: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


thetao dimensions: ('time', 'depth', 'latitude', 'longitude')
thetao shape: (366, 15, 100, 240)

Calculating sums...
Calculating squared sums...
Calculating valid counts...

Year complete.
Depth    0.0 m : valid = 0
Depth    5.0 m : valid = 4,290,252
Depth   10.0 m : valid = 4,163,616
Depth   20.0 m : valid = 4,049,058
Depth   30.0 m : valid = 3,930,474
Depth   50.0 m : valid = 3,765,408
Depth   75.0 m : valid = 3,610,224
Depth  100.0 m : valid = 3,528,240
Depth  125.0 m : valid = 3,508,110
Depth  150.0 m : valid = 3,493,104
Depth  200.0 m : valid = 3,458,700
Depth  300.0 m : valid = 3,431,616
Depth  500.0 m : valid = 3,373,788
Depth  700.0 m : valid = 3,319,986
Depth 1000.0 m : valid = 3,244,590

RAM cleaned after 2004

FINAL TRAIN-ONLY TARGET STATISTICS
   0.0 m : mean = nan   std = nan   count = 0
   5.0 m : mean = 27.995058   std = 1.709671   count = 21,416,094
  10.0 m : mean = 27.980386   std = 1.644122   count = 20,783,952
  20.0 m : mean = 27.873701   std = 1.644750   count = 2

/tmp/ipykernel_807/2441475982.py:140: RuntimeWarning: invalid value encountered in divide
  mean_theta = sum_theta / count_theta
/tmp/ipykernel_807/2441475982.py:143: RuntimeWarning: invalid value encountered in divide
  sum_theta_sq / count_theta


ValueError: Some target means are invalid.

In [ ]:
!pip install -q xarray h5netcdf netCDF4 h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.6 MB/s eta 0:00:00


In [ ]:
import xarray as xr
import os

ROOT = "/content/drive/MyDrive/OceanEmbed"
path = os.path.join(ROOT, "processed", "aligned_2000.nc")

ds = xr.open_dataset(path)

print("=" * 80)
print("GLORYS TARGET DEPTH CHECK")
print("=" * 80)

print("\nDepth coordinates:")
print(ds.depth.values)

print("\nDepth 0 m valid values:")
print(ds["thetao"].sel(depth=0).count().item())

print("\nDepth 5 m valid values:")
print(ds["thetao"].sel(depth=5).count().item())

ds.close()

GLORYS TARGET DEPTH CHECK

Depth coordinates:
[   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]

Depth 0 m valid values:
0

Depth 5 m valid values:
4290252


In [ ]:
import os
import json
import gc
import numpy as np
import xarray as xr

# ============================================================
# CONFIG
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

TRAIN_YEARS = [2000, 2001, 2002, 2003, 2004]
TEST_YEAR = 2005

TARGET_DEPTHS = np.array([
    0, 5, 10, 20, 30, 50, 75, 100,
    125, 150, 200, 300, 500, 700, 1000
], dtype=np.float32)

OUTPUT_JSON = os.path.join(
    PROCESSED,
    "target_normalization_stats.json"
)

# ============================================================
# ACCUMULATORS
# ============================================================

n_depths = len(TARGET_DEPTHS)

sum_theta = np.zeros(n_depths, dtype=np.float64)
sum_theta_sq = np.zeros(n_depths, dtype=np.float64)
count_theta = np.zeros(n_depths, dtype=np.int64)

print("=" * 80)
print("TARGET NORMALIZATION STATISTICS")
print("=" * 80)

print("Training years:", TRAIN_YEARS)
print("Test year:", TEST_YEAR)

# ============================================================
# PROCESS TRAIN YEARS
# ============================================================

for year in TRAIN_YEARS:

    path = os.path.join(
        PROCESSED,
        f"aligned_{year}.nc"
    )

    print("\n" + "=" * 80)
    print(f"PROCESSING {year}")
    print("=" * 80)

    ds = xr.open_dataset(
        path,
        chunks={
            "time": 10,
            "latitude": 25,
            "longitude": 60
        }
    )

    thetao = ds["thetao"]

    # --------------------------------------------------------
    # IMPORTANT:
    # depth 0 m has zero valid values, so skip it.
    # Calculate only depths 5-1000 m.
    # --------------------------------------------------------

    for i in range(1, n_depths):

        depth = TARGET_DEPTHS[i]

        print(f"Calculating depth: {depth:.0f} m")

        data = thetao.sel(depth=depth)

        yearly_sum = data.sum(
            dim=("time", "latitude", "longitude"),
            skipna=True
        ).compute()

        yearly_sum_sq = (data ** 2).sum(
            dim=("time", "latitude", "longitude"),
            skipna=True
        ).compute()

        yearly_count = data.count(
            dim=("time", "latitude", "longitude")
        ).compute()

        sum_theta[i] += float(yearly_sum.values)
        sum_theta_sq[i] += float(yearly_sum_sq.values)
        count_theta[i] += int(yearly_count.values)

        del data
        del yearly_sum, yearly_sum_sq, yearly_count

    ds.close()
    del ds, thetao

    gc.collect()

    print(f"Completed {year}")

# ============================================================
# CALCULATE FINAL STATISTICS
# ============================================================

mean_theta = np.full(n_depths, np.nan)
std_theta = np.full(n_depths, np.nan)

# Depth 0 intentionally remains NaN
for i in range(1, n_depths):

    mean_theta[i] = (
        sum_theta[i] / count_theta[i]
    )

    variance = (
        sum_theta_sq[i] / count_theta[i]
        - mean_theta[i] ** 2
    )

    variance = max(variance, 0.0)

    std_theta[i] = np.sqrt(variance)

# ============================================================
# PRINT
# ============================================================

print("\n" + "=" * 80)
print("FINAL TRAIN-ONLY TARGET STATISTICS")
print("=" * 80)

for i, depth in enumerate(TARGET_DEPTHS):

    if count_theta[i] == 0:

        print(
            f"{depth:6.1f} m : "
            f"NO VALID DATA"
        )

    else:

        print(
            f"{depth:6.1f} m : "
            f"mean = {mean_theta[i]:.6f}   "
            f"std = {std_theta[i]:.6f}   "
            f"count = {count_theta[i]:,}"
        )

# ============================================================
# SAVE
# ============================================================

stats = {
    "training_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "depths": TARGET_DEPTHS.tolist(),
    "mean": mean_theta.tolist(),
    "std": std_theta.tolist(),
    "count": count_theta.tolist(),
    "note": "0 m has no valid GLORYS thetao values and is intentionally left unnormalized."
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(stats, f, indent=4)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(OUTPUT_JSON)

TARGET NORMALIZATION STATISTICS
Training years: [2000, 2001, 2002, 2003, 2004]
Test year: 2005

PROCESSING 2000
Calculating depth: 5 m


/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


Calculating depth: 10 m
Calculating depth: 20 m
Calculating depth: 30 m
Calculating depth: 50 m
Calculating depth: 75 m
Calculating depth: 100 m
Calculating depth: 125 m
Calculating depth: 150 m
Calculating depth: 200 m
Calculating depth: 300 m
Calculating depth: 500 m
Calculating depth: 700 m
Calculating depth: 1000 m
Completed 2000

PROCESSING 2001
Calculating depth: 5 m


/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


Calculating depth: 10 m
Calculating depth: 20 m
Calculating depth: 30 m
Calculating depth: 50 m
Calculating depth: 75 m
Calculating depth: 100 m
Calculating depth: 125 m
Calculating depth: 150 m
Calculating depth: 200 m
Calculating depth: 300 m
Calculating depth: 500 m
Calculating depth: 700 m
Calculating depth: 1000 m
Completed 2001

PROCESSING 2002
Calculating depth: 5 m


...

Calculating depth: 10 m
Calculating depth: 20 m
Calculating depth: 30 m
Calculating depth: 50 m
Calculating depth: 75 m
Calculating depth: 100 m
Calculating depth: 125 m
Calculating depth: 150 m
Calculating depth: 200 m
Calculating depth: 300 m
Calculating depth: 500 m
Calculating depth: 700 m
Calculating depth: 1000 m
Completed 2002

PROCESSING 2003
Calculating depth: 5 m


/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


Calculating depth: 10 m
Calculating depth: 20 m
Calculating depth: 30 m
Calculating depth: 50 m
Calculating depth: 75 m
Calculating depth: 100 m
Calculating depth: 125 m
Calculating depth: 150 m
Calculating depth: 200 m
Calculating depth: 300 m
Calculating depth: 500 m
Calculating depth: 700 m
Calculating depth: 1000 m
Completed 2003

PROCESSING 2004
Calculating depth: 5 m


/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 25. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_807/1745358763.py:59: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 60. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


Calculating depth: 10 m
Calculating depth: 20 m
Calculating depth: 30 m
Calculating depth: 50 m
Calculating depth: 75 m
Calculating depth: 100 m
Calculating depth: 125 m
Calculating depth: 150 m
Calculating depth: 200 m
Calculating depth: 300 m
Calculating depth: 500 m
Calculating depth: 700 m
Calculating depth: 1000 m
Completed 2004

FINAL TRAIN-ONLY TARGET STATISTICS
   0.0 m : NO VALID DATA
   5.0 m : mean = 27.995057   std = 1.709685   count = 21,416,094
  10.0 m : mean = 27.980386   std = 1.644116   count = 20,783,952
  20.0 m : mean = 27.873701   std = 1.644750   count = 20,212,101
  30.0 m : mean = 27.616666   std = 1.747042   count = 19,620,153
  50.0 m : mean = 26.662194   std = 2.086486   count = 18,796,176
  75.0 m : mean = 24.704234   std = 2.390735   count = 18,021,528
 100.0 m : mean = 22.069137   std = 2.344613   count = 17,612,280
 125.0 m : mean = 19.454042   std = 2.145420   count = 17,511,795
 150.0 m : mean = 17.385374   std = 1.915483   count = 17,436,888
 200.0 m 

In [ ]:
# inspect the missing value
import os
import numpy as np
import xarray as xr

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

print("=" * 80)
print("OCEAN MASK / MISSING DATA CHECK")
print("=" * 80)

# Check one representative training year
YEAR = 2000

path = os.path.join(
    PROCESSED,
    f"aligned_{YEAR}.nc"
)

ds = xr.open_dataset(path)

surface_vars = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

print("\nDataset:")
print(path)

print("\nDimensions:")
print(ds.dims)

# ============================================================
# 1. SPATIAL VALIDITY OF EACH SURFACE VARIABLE
# ============================================================

print("\n" + "=" * 80)
print("SURFACE VARIABLE VALIDITY")
print("=" * 80)

for var in surface_vars:

    # Number of valid observations at every grid cell
    valid_count = ds[var].count(dim="time")

    total_days = ds.sizes["time"]

    # Cell valid for ALL days
    always_valid = (
        valid_count == total_days
    ).sum().item()

    # Cell valid for at least one day
    sometimes_valid = (
        valid_count > 0
    ).sum().item()

    total_cells = (
        ds.sizes["latitude"] *
        ds.sizes["longitude"]
    )

    print(f"\n{var}")
    print(f"  Always valid : {always_valid:,} / {total_cells:,}")
    print(f"  Sometimes valid : {sometimes_valid:,} / {total_cells:,}")

# ============================================================
# 2. COMMON SURFACE MASK
# ============================================================

print("\n" + "=" * 80)
print("COMMON SURFACE MASK")
print("=" * 80)

common_mask = None

for var in surface_vars:

    valid_count = ds[var].count(dim="time")

    # Valid for every day
    mask = valid_count == ds.sizes["time"]

    if common_mask is None:
        common_mask = mask
    else:
        common_mask = common_mask & mask

common_cells = common_mask.sum().item()

total_cells = (
    ds.sizes["latitude"] *
    ds.sizes["longitude"]
)

print(
    f"\nCommon valid cells for ALL 7 variables "
    f"and ALL {ds.sizes['time']} days:"
)

print(
    f"{common_cells:,} / {total_cells:,}"
)

print(
    f"Percentage: "
    f"{100 * common_cells / total_cells:.2f}%"
)

# ============================================================
# 3. TARGET VALIDITY BY DEPTH
# ============================================================

print("\n" + "=" * 80)
print("TARGET THETAO VALIDITY")
print("=" * 80)

for depth in ds.depth.values:

    valid_count = ds["thetao"].sel(
        depth=depth
    ).count(dim="time")

    always_valid = (
        valid_count == ds.sizes["time"]
    ).sum().item()

    sometimes_valid = (
        valid_count > 0
    ).sum().item()

    print(
        f"Depth {float(depth):6.1f} m : "
        f"always valid cells = {always_valid:,}, "
        f"sometimes valid = {sometimes_valid:,}"
    )

# ============================================================
# 4. TARGET 0 m CHECK
# ============================================================

print("\n" + "=" * 80)
print("0 m TARGET CHECK")
print("=" * 80)

zero_count = ds["thetao"].sel(depth=0).count().item()

print("Valid thetao values at 0 m:", zero_count)

if zero_count == 0:
    print(
        "0 m has NO valid GLORYS thetao values."
    )
    print(
        "We will NOT use 0 m as a supervised target."
    )

ds.close()

print("\n" + "=" * 80)
print("MASK CHECK COMPLETE")
print("=" * 80)

OCEAN MASK / MISSING DATA CHECK

Dataset:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2000.nc

Dimensions:
FrozenMappingWarningOnValuesAccess({'time': 366, 'latitude': 100, 'longitude': 240, 'depth': 15})

SURFACE VARIABLE VALIDITY

sst
  Always valid : 11,863 / 24,000
  Sometimes valid : 11,863 / 24,000

sss
  Always valid : 11,688 / 24,000
  Sometimes valid : 11,688 / 24,000

sla
  Always valid : 11,913 / 24,000
  Sometimes valid : 11,913 / 24,000

uo
  Always valid : 11,482 / 24,000
  Sometimes valid : 11,482 / 24,000

vo
  Always valid : 11,482 / 24,000
  Sometimes valid : 11,482 / 24,000

wind_u
  Always valid : 0 / 24,000
  Sometimes valid : 11,706 / 24,000

wind_v
  Always valid : 0 / 24,000
  Sometimes valid : 11,706 / 24,000

COMMON SURFACE MASK

Common valid cells for ALL 7 variables and ALL 366 days:
0 / 24,000
Percentage: 0.00%

TARGET THETAO VALIDITY
Depth    0.0 m : always valid cells = 0, sometimes valid = 0
Depth    5.0 m : always valid cells = 11,722, sometimes

In [ ]:
# we'll create a PyTorch Dataset that reads the model-ready NetCDF files on demand

import os
import json
import gc
import numpy as np
import xarray as xr

# ============================================================
# CONFIG
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

YEARS = [2000, 2001, 2002, 2003, 2004, 2005]

SURFACE_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

# 0 m removed because GLORYS thetao has no valid values there
TARGET_DEPTHS = np.array([
    5, 10, 20, 30, 50, 75, 100,
    125, 150, 200, 300, 500, 700, 1000
])

STATS_FILE = os.path.join(
    PROCESSED,
    "train_normalization_stats.json"
)

TARGET_STATS_FILE = os.path.join(
    PROCESSED,
    "target_normalization_stats.json"
)

# ============================================================
# LOAD TRAIN-ONLY STATISTICS
# ============================================================

with open(STATS_FILE, "r") as f:
    surface_stats = json.load(f)

with open(TARGET_STATS_FILE, "r") as f:
    target_stats = json.load(f)

surface_mean = {
    var: surface_stats[var]["mean"]
    for var in SURFACE_VARS
}

surface_std = {
    var: surface_stats[var]["std"]
    for var in SURFACE_VARS
}

# Target stats contain all 15 requested depths,
# but 0 m is invalid. Select only our 14 depths.

all_depths = np.array(target_stats["depths"])

target_mean_all = np.array(
    target_stats["mean"],
    dtype=np.float32
)

target_std_all = np.array(
    target_stats["std"],
    dtype=np.float32
)

depth_indices = []

for depth in TARGET_DEPTHS:
    idx = np.where(all_depths == depth)[0]

    if len(idx) != 1:
        raise ValueError(
            f"Depth {depth} m not found in target statistics."
        )

    depth_indices.append(idx[0])

target_mean = target_mean_all[depth_indices]
target_std = target_std_all[depth_indices]

# ============================================================
# PRINT CONFIG
# ============================================================

print("=" * 80)
print("CREATING MODEL-READY DATA")
print("=" * 80)

print("\nInput physical variables:")
print(SURFACE_VARS)

print("\nInput channels:")
print("7 physical + 7 validity masks = 14 channels")

print("\nTarget depths:")
print(TARGET_DEPTHS)

print("\nNumber of target depths:", len(TARGET_DEPTHS))

print("\nTrain years:", YEARS[:5])
print("Test year :", YEARS[-1])

# ============================================================
# PROCESS EACH YEAR
# ============================================================

for year in YEARS:

    input_file = os.path.join(
        PROCESSED,
        f"aligned_{year}.nc"
    )

    output_file = os.path.join(
        PROCESSED,
        f"modelready_{year}.nc"
    )

    print("\n" + "=" * 80)
    print(f"PROCESSING YEAR {year}")
    print("=" * 80)

    print("Input:")
    print(input_file)

    print("Output:")
    print(output_file)

    # --------------------------------------------------------
    # Skip if already created
    # --------------------------------------------------------

    if os.path.exists(output_file):

        print("Already exists. Skipping.")

        continue

    # --------------------------------------------------------
    # Open aligned data
    # --------------------------------------------------------

    ds = xr.open_dataset(
        input_file,
        chunks={
            "time": 10,
            "latitude": 25,
            "longitude": 60
        }
    )

    # ========================================================
    # CREATE OUTPUT DATASET
    # ========================================================

    output = xr.Dataset(
        coords={
            "time": ds.time,
            "latitude": ds.latitude,
            "longitude": ds.longitude,
            "depth": TARGET_DEPTHS
        }
    )

    # ========================================================
    # SURFACE VARIABLES
    # ========================================================

    for var in SURFACE_VARS:

        print(f"\nProcessing {var}")

        data = ds[var]

        # ----------------------------------------------------
        # Validity mask
        #
        # 1 = original observation exists
        # 0 = missing / unavailable
        # ----------------------------------------------------

        mask = xr.where(
            np.isfinite(data),
            1.0,
            0.0
        ).astype("float32")

        output[f"{var}_mask"] = mask

        # ----------------------------------------------------
        # Normalize using TRAIN ONLY statistics
        # ----------------------------------------------------

        normalized = (
            (data - surface_mean[var])
            / surface_std[var]
        )

        # ----------------------------------------------------
        # Replace NaN AFTER normalization with 0.
        #
        # IMPORTANT:
        # mask tells the model whether this 0 means
        # "normalized mean value" or "missing".
        # ----------------------------------------------------

        normalized = normalized.fillna(0.0)

        output[var] = normalized.astype("float32")

    # ========================================================
    # TARGET THETAO
    # ========================================================

    print("\nProcessing thetao target")

    thetao = ds["thetao"].sel(
        depth=TARGET_DEPTHS
    )

    # --------------------------------------------------------
    # Normalize target depth-by-depth
    #
    # NaNs remain NaN.
    # We DO NOT fill target NaNs.
    # --------------------------------------------------------

    target_mean_da = xr.DataArray(
        target_mean,
        dims=["depth"],
        coords={"depth": TARGET_DEPTHS}
    )

    target_std_da = xr.DataArray(
        target_std,
        dims=["depth"],
        coords={"depth": TARGET_DEPTHS}
    )

    thetao_normalized = (
        (thetao - target_mean_da)
        / target_std_da
    )

    output["thetao"] = thetao_normalized.astype("float32")

    # --------------------------------------------------------
    # Target validity mask
    # --------------------------------------------------------

    target_mask = xr.where(
        np.isfinite(thetao),
        1.0,
        0.0
    ).astype("float32")

    output["thetao_mask"] = target_mask

    # ========================================================
    # ATTRIBUTES
    # ========================================================

    output.attrs["description"] = (
        "OceanEmbed model-ready dataset"
    )

    output.attrs["input_channels"] = (
        "sst, sss, sla, uo, vo, wind_u, wind_v "
        "plus corresponding validity masks"
    )

    output.attrs["target"] = (
        "GLORYS thetao at 14 valid depths"
    )

    output.attrs["target_depths"] = (
        "5,10,20,30,50,75,100,125,150,200,300,500,700,1000 m"
    )

    output.attrs["surface_normalization"] = (
        "Train-only 2000-2004 mean/std"
    )

    output.attrs["target_normalization"] = (
        "Train-only 2000-2004 depth-wise mean/std"
    )

    # ========================================================
    # CHUNKING
    # ========================================================

    output = output.chunk({
        "time": 10,
        "latitude": 25,
        "longitude": 60,
        "depth": 14
    })

    # ========================================================
    # COMPRESSION
    # ========================================================

    encoding = {}

    for var in output.data_vars:

        encoding[var] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }

    # ========================================================
    # SAVE
    # ========================================================

    print("\nSaving...")

    output.to_netcdf(
        output_file,
        encoding=encoding
    )

    ds.close()

    del ds
    del output
    del thetao
    del thetao_normalized
    del target_mask

    gc.collect()

    print(f"SUCCESS: {year}")

# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("MODEL-READY DATA CREATION COMPLETE")
print("=" * 80)

for year in YEARS:

    path = os.path.join(
        PROCESSED,
        f"modelready_{year}.nc"
    )

    if os.path.exists(path):

        size_gb = os.path.getsize(path) / (1024 ** 3)

        print(
            f"{year}: EXISTS | "
            f"{size_gb:.2f} GB"
        )

    else:

        print(f"{year}: MISSING")

print("\nNext step:")
print("Create training/validation/test sequences.")

CREATING MODEL-READY DATA

Input physical variables:
['sst', 'sss', 'sla', 'uo', 'vo', 'wind_u', 'wind_v']

Input channels:
7 physical + 7 validity masks = 14 channels

Target depths:
[   5   10   20   30   50   75  100  125  150  200  300  500  700 1000]

Number of target depths: 14

Train years: [2000, 2001, 2002, 2003, 2004]
Test year : 2005

PROCESSING YEAR 2000
Input:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2000.nc
Output:
/content/drive/MyDrive/OceanEmbed/processed/modelready_2000.nc
Already exists. Skipping.

PROCESSING YEAR 2001
Input:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2001.nc
Output:
/content/drive/MyDrive/OceanEmbed/processed/modelready_2001.nc
Already exists. Skipping.

PROCESSING YEAR 2002
Input:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2002.nc
Output:
/content/drive/MyDrive/OceanEmbed/processed/modelready_2002.nc
Already exists. Skipping.

PROCESSING YEAR 2003
Input:
/content/drive/MyDrive/OceanEmbed/processed/aligned_2003.nc
Output

In [ ]:

import os
import xarray as xr
import numpy as np

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

YEARS = [2000, 2001, 2002, 2003, 2004, 2005]

print("=" * 80)
print("VERIFYING MODEL-READY DATA")
print("=" * 80)

for year in YEARS:

    path = os.path.join(
        PROCESSED,
        f"modelready_{year}.nc"
    )

    print("\n" + "-" * 80)
    print(f"YEAR {year}")

    ds = xr.open_dataset(path)

    print("Dimensions:")
    print(dict(ds.sizes))

    print("\nVariables:")
    print(list(ds.data_vars))

    # --------------------------------------------------------
    # Check expected input variables
    # --------------------------------------------------------

    expected_surface = [
        "sst", "sss", "sla",
        "uo", "vo",
        "wind_u", "wind_v"
    ]

    expected_masks = [
        "sst_mask", "sss_mask", "sla_mask",
        "uo_mask", "vo_mask",
        "wind_u_mask", "wind_v_mask"
    ]

    expected_target = [
        "thetao",
        "thetao_mask"
    ]

    missing = []

    for var in (
        expected_surface
        + expected_masks
        + expected_target
    ):
        if var not in ds:
            missing.append(var)

    if missing:
        print("ERROR: Missing variables:", missing)
    else:
        print("All expected variables present.")

    # --------------------------------------------------------
    # Check shapes
    # --------------------------------------------------------

    print("\nSurface shape:")
    print(ds["sst"].shape)

    print("Target shape:")
    print(ds["thetao"].shape)

    # --------------------------------------------------------
    # Check depth
    # --------------------------------------------------------

    print("\nTarget depths:")
    print(ds.depth.values)

    # --------------------------------------------------------
    # Check NaNs in input
    # --------------------------------------------------------

    print("\nInput NaN check:")

    for var in expected_surface:

        nan_count = int(
            ds[var].isnull().sum().compute()
        )

        print(
            f"  {var:8s}: NaNs = {nan_count:,}"
        )

    # --------------------------------------------------------
    # Check target
    # --------------------------------------------------------

    print("\nTarget NaN check:")

    for depth in ds.depth.values:

        valid = int(
            ds["thetao"]
            .sel(depth=depth)
            .count()
            .compute()
        )

        print(
            f"  {float(depth):6.1f} m : "
            f"valid = {valid:,}"
        )

    ds.close()

print("\n" + "=" * 80)
print("MODEL-READY VERIFICATION COMPLETE")
print("=" * 80)

VERIFYING MODEL-READY DATA

--------------------------------------------------------------------------------
YEAR 2000
Dimensions:
{'time': 366, 'latitude': 100, 'longitude': 240, 'depth': 14}

Variables:
['sst_mask', 'sst', 'sss_mask', 'sss', 'sla_mask', 'sla', 'uo_mask', 'uo', 'vo_mask', 'vo', 'wind_u_mask', 'wind_u', 'wind_v_mask', 'wind_v', 'thetao', 'thetao_mask']
All expected variables present.

Surface shape:
(366, 100, 240)
Target shape:
(366, 14, 100, 240)

Target depths:
[   5   10   20   30   50   75  100  125  150  200  300  500  700 1000]

Input NaN check:
  sst     : NaNs = 0
  sss     : NaNs = 0
  sla     : NaNs = 0
  uo      : NaNs = 0
  vo      : NaNs = 0
  wind_u  : NaNs = 0
  wind_v  : NaNs = 0

Target NaN check:
     5.0 m : valid = 4,290,252
    10.0 m : valid = 4,163,616
    20.0 m : valid = 4,049,058
    30.0 m : valid = 3,930,474
    50.0 m : valid = 3,765,408
    75.0 m : valid = 3,610,224
   100.0 m : valid = 3,528,240
   125.0 m : valid = 3,508,110
   150.0 m

In [ ]:
# ============================================================
# OCEAN EMBED - SEQUENCE DATASET
# ============================================================

import os
import json
import numpy as np
import xarray as xr
import torch
from torch.utils.data import Dataset

# ============================================================
# CONFIG
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

INPUT_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

MASK_VARS = [
    "sst_mask",
    "sss_mask",
    "sla_mask",
    "uo_mask",
    "vo_mask",
    "wind_u_mask",
    "wind_v_mask"
]

TARGET_VAR = "thetao"
TARGET_MASK = "thetao_mask"

SEQUENCE_LENGTH = 3

PATCH_SIZE = 32
PATCH_STRIDE = 32

TARGET_DEPTHS = [
    5, 10, 20, 30, 50, 75, 100,
    125, 150, 200, 300, 500, 700, 1000
]

# ============================================================
# YEARS
# ============================================================

TRAIN_YEARS = [2000, 2001, 2002, 2003]
VAL_YEARS = [2004]
TEST_YEARS = [2005]

# ============================================================
# DATASET
# ============================================================

class OceanSequenceDataset(Dataset):

    def __init__(
        self,
        years,
        root,
        sequence_length=3,
        patch_size=32,
        stride=32
    ):

        self.years = years
        self.root = root
        self.sequence_length = sequence_length
        self.patch_size = patch_size
        self.stride = stride

        self.datasets = {}
        self.samples = []

        # ----------------------------------------------------
        # Open datasets
        # ----------------------------------------------------

        print("\nOpening datasets...")

        for year in years:

            path = os.path.join(
                root,
                f"modelready_{year}.nc"
            )

            print(f"  {year}: {path}")

            ds = xr.open_dataset(
                path
            )

            self.datasets[year] = ds

            ntime = ds.sizes["time"]
            nlat = ds.sizes["latitude"]
            nlon = ds.sizes["longitude"]

            # ------------------------------------------------
            # Calculate patch locations
            # ------------------------------------------------

            lat_positions = list(
                range(
                    0,
                    nlat - patch_size + 1,
                    stride
                )
            )

            lon_positions = list(
                range(
                    0,
                    nlon - patch_size + 1,
                    stride
                )
            )

            # ------------------------------------------------
            # Sequence samples
            #
            # t-2, t-1, t -> target t
            # ------------------------------------------------

            for t in range(
                sequence_length - 1,
                ntime
            ):

                for lat in lat_positions:

                    for lon in lon_positions:

                        self.samples.append(
                            (
                                year,
                                t,
                                lat,
                                lon
                            )
                        )

        print("\nDataset created.")

        print(
            "Number of samples:",
            len(self.samples)
        )

        print(
            "Patch size:",
            patch_size,
            "x",
            patch_size
        )

        print(
            "Sequence length:",
            sequence_length
        )

    # ========================================================
    # LENGTH
    # ========================================================

    def __len__(self):

        return len(self.samples)

    # ========================================================
    # GET ITEM
    # ========================================================

    def __getitem__(self, idx):

        year, t, lat, lon = self.samples[idx]

        ds = self.datasets[year]

        # ----------------------------------------------------
        # Input
        # ----------------------------------------------------

        input_days = []

        for day in range(
            t - self.sequence_length + 1,
            t + 1
        ):

            channels = []

            # ----------------------------------------------
            # Physical variables
            # ----------------------------------------------

            for var in INPUT_VARS:

                data = ds[var].isel(
                    time=day,
                    latitude=slice(
                        lat,
                        lat + self.patch_size
                    ),
                    longitude=slice(
                        lon,
                        lon + self.patch_size
                    )
                ).values

                channels.append(
                    data.astype(np.float32)
                )

            # ----------------------------------------------
            # Validity masks
            # ----------------------------------------------

            for var in MASK_VARS:

                data = ds[var].isel(
                    time=day,
                    latitude=slice(
                        lat,
                        lat + self.patch_size
                    ),
                    longitude=slice(
                        lon,
                        lon + self.patch_size
                    )
                ).values

                channels.append(
                    data.astype(np.float32)
                )

            # ----------------------------------------------
            # (14, H, W)
            # ----------------------------------------------

            channels = np.stack(
                channels,
                axis=0
            )

            input_days.append(channels)

        # ----------------------------------------------------
        # (3, 14, H, W)
        # ----------------------------------------------------

        X = np.stack(
            input_days,
            axis=0
        )

        # ----------------------------------------------------
        # Target
        # ----------------------------------------------------

        Y = ds[TARGET_VAR].isel(
            time=t,
            depth=slice(None),
            latitude=slice(
                lat,
                lat + self.patch_size
            ),
            longitude=slice(
                lon,
                lon + self.patch_size
            )
        ).values.astype(np.float32)

        # ----------------------------------------------------
        # Target mask
        # ----------------------------------------------------

        M = ds[TARGET_MASK].isel(
            time=t,
            depth=slice(None),
            latitude=slice(
                lat,
                lat + self.patch_size
            ),
            longitude=slice(
                lon,
                lon + self.patch_size
            )
        ).values.astype(np.float32)

        # ----------------------------------------------------
        # Replace target NaNs with 0.
        #
        # IMPORTANT:
        # M tells the loss which pixels are valid.
        # ----------------------------------------------------

        Y = np.nan_to_num(
            Y,
            nan=0.0
        )

        # ----------------------------------------------------
        # Convert to Torch
        # ----------------------------------------------------

        X = torch.from_numpy(X)
        Y = torch.from_numpy(Y)
        M = torch.from_numpy(M)

        return X, Y, M

In [ ]:
# ============================================================
# OCEANEMBED - SEQUENCE DATASET
# ============================================================

import os
import numpy as np
import xarray as xr
import torch
from torch.utils.data import Dataset

# ============================================================
# PATHS
# ============================================================

ROOT = "/content/drive/MyDrive/OceanEmbed"
PROCESSED = os.path.join(ROOT, "processed")

# ============================================================
# INPUT CONFIG
# ============================================================

INPUT_VARS = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "wind_u",
    "wind_v"
]

MASK_VARS = [
    "sst_mask",
    "sss_mask",
    "sla_mask",
    "uo_mask",
    "vo_mask",
    "wind_u_mask",
    "wind_v_mask"
]

TARGET_VAR = "thetao"
TARGET_MASK_VAR = "thetao_mask"

SEQUENCE_LENGTH = 3

PATCH_SIZE = 32
PATCH_STRIDE = 32

TARGET_DEPTHS = [
    5, 10, 20, 30, 50, 75, 100,
    125, 150, 200, 300, 500, 700, 1000
]

# ============================================================
# YEAR SPLIT
# ============================================================

TRAIN_YEARS = [2000, 2001, 2002, 2003]
VAL_YEARS   = [2004]
TEST_YEARS  = [2005]

# ============================================================
# DATASET CLASS
# ============================================================

class OceanSequenceDataset(Dataset):

    def __init__(
        self,
        years,
        root,
        sequence_length=3,
        patch_size=32,
        stride=32
    ):

        self.years = years
        self.root = root
        self.sequence_length = sequence_length
        self.patch_size = patch_size
        self.stride = stride

        self.datasets = {}
        self.samples = []

        # ----------------------------------------------------
        # OPEN FILES
        # ----------------------------------------------------

        print("\nOpening datasets...")

        for year in years:

            path = os.path.join(
                root,
                f"modelready_{year}.nc"
            )

            print(f"  Opening {year}")

            ds = xr.open_dataset(
                path
            )

            self.datasets[year] = ds

            ntime = ds.sizes["time"]
            nlat = ds.sizes["latitude"]
            nlon = ds.sizes["longitude"]

            # ------------------------------------------------
            # PATCH POSITIONS
            # ------------------------------------------------

            lat_positions = range(
                0,
                nlat - patch_size + 1,
                stride
            )

            lon_positions = range(
                0,
                nlon - patch_size + 1,
                stride
            )

            lat_positions = list(lat_positions)
            lon_positions = list(lon_positions)

            print(
                f"    Time steps : {ntime}"
            )

            print(
                f"    Lat patches: {len(lat_positions)}"
            )

            print(
                f"    Lon patches: {len(lon_positions)}"
            )

            # ------------------------------------------------
            # CREATE SAMPLE INDEX
            #
            # t-2, t-1, t -> target t
            # ------------------------------------------------

            for t in range(
                sequence_length - 1,
                ntime
            ):

                for lat in lat_positions:

                    for lon in lon_positions:

                        self.samples.append(
                            (
                                year,
                                t,
                                lat,
                                lon
                            )
                        )

        # ----------------------------------------------------
        # INFORMATION
        # ----------------------------------------------------

        print("\nDataset ready.")

        print(
            "Years:",
            years
        )

        print(
            "Sequence length:",
            sequence_length
        )

        print(
            "Patch:",
            f"{patch_size} x {patch_size}"
        )

        print(
            "Total samples:",
            len(self.samples)
        )

    # ========================================================
    # LENGTH
    # ========================================================

    def __len__(self):

        return len(self.samples)

    # ========================================================
    # GET ITEM
    # ========================================================

    def __getitem__(self, index):

        year, t, lat, lon = self.samples[index]

        ds = self.datasets[year]

        # ====================================================
        # INPUT SEQUENCE
        # ====================================================

        sequence = []

        start_day = t - self.sequence_length + 1

        for day in range(
            start_day,
            t + 1
        ):

            channels = []

            # ------------------------------------------------
            # Physical variables
            # ------------------------------------------------

            for var in INPUT_VARS:

                data = ds[var].isel(
                    time=day,
                    latitude=slice(
                        lat,
                        lat + self.patch_size
                    ),
                    longitude=slice(
                        lon,
                        lon + self.patch_size
                    )
                ).values

                channels.append(
                    np.asarray(
                        data,
                        dtype=np.float32
                    )
                )

            # ------------------------------------------------
            # Validity masks
            # ------------------------------------------------

            for var in MASK_VARS:

                data = ds[var].isel(
                    time=day,
                    latitude=slice(
                        lat,
                        lat + self.patch_size
                    ),
                    longitude=slice(
                        lon,
                        lon + self.patch_size
                    )
                ).values

                channels.append(
                    np.asarray(
                        data,
                        dtype=np.float32
                    )
                )

            # ------------------------------------------------
            # 14 channels
            # ------------------------------------------------

            channels = np.stack(
                channels,
                axis=0
            )

            sequence.append(channels)

        # ====================================================
        # FINAL INPUT
        # ====================================================

        X = np.stack(
            sequence,
            axis=0
        )

        # Shape:
        # (3, 14, 32, 32)

        # ====================================================
        # TARGET
        # ====================================================

        Y = ds[TARGET_VAR].isel(
            time=t,
            depth=slice(None),
            latitude=slice(
                lat,
                lat + self.patch_size
            ),
            longitude=slice(
                lon,
                lon + self.patch_size
            )
        ).values

        Y = np.asarray(
            Y,
            dtype=np.float32
        )

        # ====================================================
        # TARGET MASK
        # ====================================================

        M = ds[TARGET_MASK_VAR].isel(
            time=t,
            depth=slice(None),
            latitude=slice(
                lat,
                lat + self.patch_size
            ),
            longitude=slice(
                lon,
                lon + self.patch_size
            )
        ).values

        M = np.asarray(
            M,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Target NaNs -> 0
        #
        # M tells the loss which pixels are valid.
        # ----------------------------------------------------

        Y = np.nan_to_num(
            Y,
            nan=0.0
        )

        # ====================================================
        # TORCH
        # ====================================================

        X = torch.from_numpy(X)
        Y = torch.from_numpy(Y)
        M = torch.from_numpy(M)

        return X, Y, M

In [ ]:
# ============================================================
# CREATE DATASETS
# ============================================================

print("=" * 80)
print("TRAIN DATASET")
print("=" * 80)

train_dataset = OceanSequenceDataset(
    years=TRAIN_YEARS,
    root=PROCESSED,
    sequence_length=3,
    patch_size=32,
    stride=32
)

print("\n" + "=" * 80)
print("VALIDATION DATASET")
print("=" * 80)

val_dataset = OceanSequenceDataset(
    years=VAL_YEARS,
    root=PROCESSED,
    sequence_length=3,
    patch_size=32,
    stride=32
)

print("\n" + "=" * 80)
print("TEST DATASET")
print("=" * 80)

test_dataset = OceanSequenceDataset(
    years=TEST_YEARS,
    root=PROCESSED,
    sequence_length=3,
    patch_size=32,
    stride=32
)

print("\n" + "=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

print("Train samples     :", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples      :", len(test_dataset))

TRAIN DATASET

Opening datasets...
  Opening 2000
    Time steps : 366
    Lat patches: 3
    Lon patches: 7
  Opening 2001
    Time steps : 365
    Lat patches: 3
    Lon patches: 7
  Opening 2002
    Time steps : 365
    Lat patches: 3
    Lon patches: 7
  Opening 2003
    Time steps : 365
    Lat patches: 3
    Lon patches: 7

Dataset ready.
Years: [2000, 2001, 2002, 2003]
Sequence length: 3
Patch: 32 x 32
Total samples: 30513

VALIDATION DATASET

Opening datasets...
  Opening 2004
    Time steps : 366
    Lat patches: 3
    Lon patches: 7

Dataset ready.
Years: [2004]
Sequence length: 3
Patch: 32 x 32
Total samples: 7644

TEST DATASET

Opening datasets...
  Opening 2005
    Time steps : 365
    Lat patches: 3
    Lon patches: 7

Dataset ready.
Years: [2005]
Sequence length: 3
Patch: 32 x 32
Total samples: 7623

DATASET SUMMARY
Train samples     : 30513
Validation samples: 7644
Test samples      : 7623


In [ ]:
# ============================================================
# TEST ONE SAMPLE
# ============================================================

X, Y, M = train_dataset[0]

print("=" * 80)
print("SINGLE SAMPLE CHECK")
print("=" * 80)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("M shape:", M.shape)

print("\nExpected:")
print("X = (3, 14, 32, 32)")
print("Y = (14, 32, 32)")
print("M = (14, 32, 32)")

print("\nX dtype:", X.dtype)
print("Y dtype:", Y.dtype)
print("M dtype:", M.dtype)

print("\nX NaNs:", torch.isnan(X).sum().item())
print("Y NaNs:", torch.isnan(Y).sum().item())
print("M NaNs:", torch.isnan(M).sum().item())

print("\nX min:", X.min().item())
print("X max:", X.max().item())

print("\nY min:", Y.min().item())
print("Y max:", Y.max().item())

print("\nMask unique values:")
print(torch.unique(M))

SINGLE SAMPLE CHECK
X shape: torch.Size([3, 14, 32, 32])
Y shape: torch.Size([14, 32, 32])
M shape: torch.Size([14, 32, 32])

Expected:
X = (3, 14, 32, 32)
Y = (14, 32, 32)
M = (14, 32, 32)

X dtype: torch.float32
Y dtype: torch.float32
M dtype: torch.float32

X NaNs: 0
Y NaNs: 0
M NaNs: 0

X min: -3.8766610622406006
X max: 2.7759432792663574

Y min: -3.7787647247314453
Y max: 3.7820165157318115

Mask unique values:
tensor([0., 1.])


In [ ]:
# ============================================================
# OCEANEMBED - DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BATCH_SIZE = 16
NUM_WORKERS = 0

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

print("=" * 80)
print("DATALOADERS CREATED")
print("=" * 80)

print("Batch size:", BATCH_SIZE)

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DATALOADERS CREATED
Batch size: 16

Train batches: 1908
Validation batches: 478
Test batches: 477


In [ ]:
# ============================================================
# BATCH SHAPE CHECK
# ============================================================

print("=" * 80)
print("LOADING ONE TRAINING BATCH")
print("=" * 80)

X_batch, Y_batch, M_batch = next(iter(train_loader))

print("\nX batch shape:")
print(X_batch.shape)

print("\nY batch shape:")
print(Y_batch.shape)

print("\nM batch shape:")
print(M_batch.shape)

print("\nDtypes:")
print("X:", X_batch.dtype)
print("Y:", Y_batch.dtype)
print("M:", M_batch.dtype)

print("\nNaN check:")
print("X NaNs:", torch.isnan(X_batch).sum().item())
print("Y NaNs:", torch.isnan(Y_batch).sum().item())
print("M NaNs:", torch.isnan(M_batch).sum().item())

print("\nMask values:")
print(torch.unique(M_batch))

print("\nMemory:")
print(
    f"X batch memory: "
    f"{X_batch.numel() * 4 / 1024**2:.2f} MB"
)

print("=" * 80)
print("BATCH CHECK COMPLETE")
print("=" * 80)

LOADING ONE TRAINING BATCH

X batch shape:
torch.Size([16, 3, 14, 32, 32])

Y batch shape:
torch.Size([16, 14, 32, 32])

M batch shape:
torch.Size([16, 14, 32, 32])

Dtypes:
X: torch.float32
Y: torch.float32
M: torch.float32

NaN check:
X NaNs: 0
Y NaNs: 0
M NaNs: 0

Mask values:
tensor([0., 1.])

Memory:
X batch memory: 2.62 MB
BATCH CHECK COMPLETE


# SWItching to the gpu for training

In [ ]:
import torch

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
else:
    print("WARNING: GPU NOT AVAILABLE")

print("=" * 70)

GPU CHECK
PyTorch version: 2.11.0+cpu
CUDA available: False
